# Research2: from particles to a figure

A local, optional teaching notebook. **Synthetic controls are explicit.** A plot is
not proof of a resolved galaxy, a converged estimator or an accepted posterior.
This notebook explains the quick lab, reads its actual saved arrays and exposes
all registered scientific recipes with their original input contracts.

Start the dashboard with `python optional_tests/lab.py serve`, run the 32-particle
lab, and then run these cells. No cell downloads data or creates a video.


In [ ]:
from pathlib import Path
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image
base = Path.cwd().resolve()
ROOT = next(p for p in [base, *base.parents] if (p / 'configs/research/graph_producers.json').is_file())
sys.path.insert(0, str(ROOT))
from optional_tests.catalogue import build_catalogue
catalogue = build_catalogue()
print(catalogue['counts'])


## Select a fresh lab run

Set `RESEARCH2_LAB_RUN` to an explicit results directory, or let the cell select
the newest `run.json` below the optional-lab runtime root. A missing run leaves
an actionable message; archived gallery examples can still be inspected below.


In [ ]:
specified = os.environ.get('RESEARCH2_LAB_RUN')
choices = list((ROOT.parent / 'research2-runtime/optional-lab').rglob('run.json')) if not specified else []
RUN = Path(specified).resolve() if specified else (max(choices, key=lambda p: p.stat().st_mtime).parent if choices else None)
if RUN is None:
    print('No fresh lab output. Run the dashboard, or call run_lab below with a NEW directory.')
else:
    receipt = json.loads((RUN / 'run.json').read_text())
    print(RUN, receipt['status'], receipt['particles'], 'particles')
    for step in receipt['steps']:
        print(step['id'], step['status'], step.get('seconds'), 'seconds')


To create a new run directly, use a fresh destination:

```python
from optional_tests.lab import run_lab
run_lab(ROOT.parent / 'research2-runtime/optional-lab/my-new-32', particles=32)
```

`32` is the **total** count in this lab. In cosmological IC configuration,
`particles_per_dimension=32` instead means $32^3$ particles per generated species.


## Which Python files actually ran?

Fresh runs include a main-thread call record. It separates function calls from
module imports and class definitions, and binds the source hashes captured at
the start of the measurements. These counts are not full branch coverage.
Native code and the isolated library workers have separate receipts. The next
cell checks the recorded trace against the run receipt, then lists the actual
observed functions and whether their captured source still matches this checkout.


In [ ]:
trace_path = RUN / 'python-calls.json' if RUN is not None else None
if trace_path is None or not trace_path.is_file():
    print('This run predates per-file observation. Run a fresh lab profile for observed calls.')
else:
    from hashlib import sha256
    recorded = {row['path']: row for row in receipt['artifacts']}
    assert sha256(trace_path.read_bytes()).hexdigest() == recorded['python-calls.json']['sha256']
    observed = json.loads(trace_path.read_text())
    assert observed == receipt['python_execution']
    print(observed['scope'])
    print(observed['files_with_calls'], 'files with calls;', observed['call_events'], 'function call events')
    for name, entry in sorted(observed['files'].items()):
        path = (ROOT / name).resolve()
        matches = path.is_relative_to(ROOT) and path.is_file() and sha256(path.read_bytes()).hexdigest() == entry['source_sha256']
        print(name, '| calls:', sum(entry['calls'].values()), '| imports:', entry['module_events'],
              '| class definitions:', entry['definition_events'], '| current source matches:', matches)
        for function, count in sorted(entry['calls'].items()):
            print('  ', count, function)


## Known flyby and event-time reconstruction

`r(t)² = v²(t−t_peri)² + b²; t_infall = t_peri − sqrt(R²−b²)/v`

**Units.** Proper separation, impact parameter b and boundary R: kpc; time: Myr; proper speed: kpc/Myr (converted to/from peculiar km/s with H x).

**Reading the result.** The 32-particle authored flyby has three saved epochs, a fixed 1.5-kpc boundary, b=0.5 kpc and v=0.4 kpc/Myr. The original graph/image/clock pipeline recovers candidate infall and pericentre times. The saved brackets and interpolation-uncertified status remain; this is not a physical merger or finder detection.

Implementation: `core/assembly.py`, `orbital_assembly_events`.

## Known-shape recovery from mass moments

`I_ij = Σ m x_i x_j / Σ m; b/a = sqrt(λ1/λ0); c/a = sqrt(λ2/λ0); θ = acos(|e_inner · e_outer|)`

**Units.** Offsets: proper kpc; tensor: kpc²; mass: Msun; ratios: dimensionless; unsigned twist: degrees.

**Reading the result.** The 6000-point authored triaxial shells have known ratios and rotations. Iterative ellipsoidal selection and bootstrap orientation support must pass before displaying the twist. A fixed centre and complete aperture are retained. This is geometric recovery, not halo convergence.

Implementation: `core/halos.py`, `radial_density_shapes`.

## Known density valley and correlated derivative uncertainty

`s = D lnρ; C_s = D C_lnρ Dᵀ; width = ln(r_outer/r_inner) at half prominence`

**Units.** Radius: proper kpc; shell density: Msun/kpc³; slope, prominence and log-radius width: dimensionless.

**Reading the result.** The 32768-particle authored profile has a known valley. The primary seven-bin GLS derivative keeps missing bins and its full propagated covariance. Correlated Gaussian covariance draws measure conditional feature support; they are not independent particle simulations. Five- and nine-bin alternatives stay separately recorded.

Implementation: `core/analysis.py`, `uncertainty_aware_log_density_derivative`.

## Native gas energy and implicit cooling

`u_next = u_before + Δt L_net(T_next, composition_next) / ρ; u_cgs = u_saved × velocity_unit²`

**Units.** u: erg/g; net volumetric heating/cooling L_net: erg/(cm³ s); ρ: g/cm³; source Δt: s.

**Reading the result.** The gas control evolves hydrodynamics and native atomic cooling. Its mass-weighted specific energy includes pressure work as well as the cooling source. Read the original energy arrays and unit attributes. This is a short 64-gas execution control; it does not establish cooling convergence.

Implementation: `core/evolution.py`, `gadget_param_text`.

## Native SH03 stellar birth opportunities

`t_star(ρ) = t0 sqrt(ρ_threshold / ρ); dotρ_locked = (1 − β) x ρ / t_star`

**Units.** ρ: proper gas density; t_star: proper time; β: prompt returned fraction; x: cold-cloud mass fraction.

**Reading the result.** The native source samples stellar births with exact particle IDs, counters and StellarRandomSeed. This control starts with no stars. New SSPBirthMode=1 records native SH03 births; seeded mode 2 is distinct. Gas energy on the effective multiphase EOS is not a single-phase temperature. The gas/star census and source journal are reconciled by the production history audit. Delayed stellar returns are disabled.

Implementation: `core/evolution.py`, `hydro_star_formation_parameters`.

## Native seeded black-hole accretion

`dotM_in = min[α f_rot 4πG² M_BH² ρ/(c_s²+v_rel²)^(3/2), dotM_Edd]; dotM_BH = (1−ε_r) dotM_in; ΔE_feedback = ε_f ε_r dotM_in c² Δt`

**Units.** Physical BH mass: Msun; inflow: Msun/year; density: g/cm³; velocities: cm/s in the native source. Saved comoving Header/Time is scale factor a.

**Reading the result.** One 10-Msun BH is explicitly supplied with 64 gas particles. The analytic gas IC is compressed before evolution, and the original sealed grid remains retained. Native fields show physical mass and inflow separately from gravitational particle mass. Positive growth/inflow checks execution; full source-budget reconciliation and resolved BH accuracy are separate requirements.

Implementation: `core/evolution.py`, `gadget_param_text`.

## CLASS power and dimensionless particle handoff

`Δ²(k,z) = k³ P_CLASS(k,z) / (2π²); handoff columns = log10(k), log10(Δ²)`

**Units.** k: h/Mpc; P_CLASS: (Mpc/h)³; Δ²: dimensionless. The handoff uses the saved start redshift.

**Reading the result.** These arrays come from actual linear Boltzmann solver outputs, not from the quick lab particle FFT. CLASS and AxiCLASS run separately at z=49 and z=0. Inspect the original model, density/velocity transfer gauge, scalar-field scope, source hashes and background conventions before preparing particles. A completed spectrum is not a nonlinear simulation or a calibrated cosmological constraint.

Implementation: `core/linear.py`, `export_power`.

## Calibrated parameter-response modes

`Jᵢⱼ = ∂yᵢ/∂θⱼ; C = L Lᵀ; A = L⁻¹ J diag(s); A = U diag(σ) Vᵀ`

**Units.** J has observable units divided by parameter units. C has squared observable units. Each positive s has its parameter units; A, singular values σ and right-mode loadings are dimensionless.

**Reading the result.** Independent held-out forward runs must first check the local response. Large singular values identify locally informative scaled parameter combinations; zero values retain unresolved combinations. The 32-particle BH control measures final BH mass, uses scales 10⁻²⁰ cm²/g and 10,000 Msun, and an explicit covariance floor of 10,000 Msun². Its singular values are 100 and 0: initial mass is visible, scattering is unconstrained. This is not a feedback measurement or a calibrated posterior. Mode signs are arbitrary; read parameter order and scales from the original product.

Implementation: `core/sidm.py`, `forward_model_degeneracy`.

## Density from actual mass in shells

`ρⱼ = Σᵢ∈shell(j) mᵢ / [4π(rⱼ₊₁³ − rⱼ³)/3]`

**Units.** m: Msun; r: proper kpc; ρ: Msun / proper kpc³

**Reading the result.** Shell averages are not pointwise density. Empty shells have measured zero mass; log plots mask zero. At 32 particles the sampling noise is large.

Implementation: `optional_tests/lab.py`, `measure_density`.

For shell $j$ with edges $r_j,r_{j+1}$,
$$V_j=\frac{4\pi}{3}(r_{j+1}^3-r_j^3),\qquad
\rho_j=\frac{\sum_{i\in j}m_i}{V_j}.$$
The plotted radius is the arithmetic midpoint; it is not a new measurement
of a pointwise density. The surface map divides pixel mass by pixel area.
The mask used on the logarithmic density axis is saved alongside the counts.


In [ ]:
if RUN is not None:
    with np.load(RUN / 'density_arrays.npz') as data:
        rho = data['shell_mass_msun'] / data['shell_volume_kpc3']
        np.testing.assert_allclose(rho, data['density_msun_kpc3'])
        print('Measured total mass [Msun]:', data['shell_mass_msun'].sum())
        print('Shell counts:', data['shell_count'])
        support = data['supported']
        plt.loglog(data['radius_proper_kpc'][support], rho[support], 'o-')
        plt.xlabel('Shell midpoint [proper kpc]'); plt.ylabel('Density [Msun / proper kpc³]')
        plt.show()
    display(Image(filename=str(RUN / 'density.png')))


## Periodic mesh power

`δ = Ncell/⟨Ncell⟩ − 1; P(k) = V |FFT(δ)|² / Nmesh⁶`

**Units.** In this lab: k in 1/proper kpc; P in proper kpc³. Cosmological registered products use their saved Mpc and h conventions.

**Reading the result.** The lab uses NGP assignment and includes sampling/assignment effects. It is not the native GADGET folded, CIC-deconvolved, shot-noise-subtracted spectrum.

Implementation: `core/analysis.py`, `fft_power_spectrum`.

In [ ]:
if RUN is not None:
    with np.load(RUN / 'power_arrays.npz') as data:
        plt.loglog(data['k_inverse_proper_kpc'], data['power_proper_kpc3'], 'o-')
        plt.xlabel('k [1 / proper kpc]'); plt.ylabel('P(k) [proper kpc³]'); plt.show()
    # Read the implementation to see rFFT multiplicity and normalization.
    from core.analysis import fft_power_spectrum
    import inspect
    print(inspect.getsource(fft_power_spectrum))


## Physical orbit integration

`dx/dt = v; dv/dt = −∇Φ; E = |v|²/2 + Φ`

**Units.** x: proper kpc; v: km/s; t: Myr; Φ and E: (km/s)²; code converts km/s to kpc/Myr.

**Reading the result.** The quick lab integrates tracers in a declared static isochrone potential. Tracers have no self-gravity and are not a native galaxy simulation.

Implementation: `core/streams.py`, `integrate_stream`.

In [ ]:
if RUN is not None:
    with np.load(RUN / 'orbit_arrays.npz') as data:
        E = data['specific_energy_kms2']
        drift = np.max(np.abs(E-E[0]) / np.maximum(np.abs(E[0]), 1.0))
        print('Max relative finite-tracer energy drift:', drift)
        print('Actual saved Myr:', data['times_myr'])
    display(Image(filename=str(RUN / 'orbits.png')))


## Spherical-equivalent circular speed

`v_c(r) = √[G M(<r)/r]`

**Units.** G = 4.3009… × 10⁻⁶ proper kpc (km/s)² / Msun

**Reading the result.** This is a spherical-equivalent diagnostic. It is not necessarily the rotation speed of a nonspherical system.

Implementation: `core/halo_products.py`, `export_native_structure_figures`.

## Weighted population velocity covariance

`pᵢ = mᵢ/Σm; μ = Σpᵢvᵢ; C = Σpᵢ(vᵢ−μ)(vᵢ−μ)ᵀ`

**Units.** μ: km/s; C: (km/s)²; effective count: 1/Σpᵢ²

**Reading the result.** This describes the finite measured population, not an unbiased covariance estimate of an unknown population or the covariance of the mean.

Implementation: `core/stream_common.py`, `weighted_covariance`.

In [ ]:
if RUN is not None:
    with np.load(RUN / 'velocity_arrays.npz') as data:
        print('Mean [km/s]:', data['mean_velocity_kms'])
        C = data['population_covariance_kms2']
        print('Covariance [(km/s)²]:', C)
        print('Eigenvalues:', np.linalg.eigvalsh(C))


## Native to physical units

`physical = saved × to_cgs × a^(a_scaling) × h^(h_scaling)`

**Units.** The synthetic lab uses isolated proper coordinates, h=1, zero a/h exponents. Cosmological data require their original conventions.

**Reading the result.** Mass, position, velocity and clock conversions are separate. Header/Time is elapsed native time for an isolated file, not a scale factor.

Implementation: `core/data.py`, `read_species_phase_space`.

In [ ]:
if RUN is not None:
    import h5py
    with h5py.File(RUN / 'synthetic_snapshot.hdf5') as f:
        print('Origin:', f.attrs['source_origin'])
        print('Clock:', f['Parameters'].attrs['ComovingIntegrationOn'], '(0 = isolated)')
        for name in ['PartType0','PartType1','PartType4','PartType5']:
            print(name, len(f[name]['ParticleIDs']), dict(f[name]['Coordinates'].attrs))


## Simulation-based calibration ranks

`rⱼ = count(θⱼ,draw < θⱼ,true) + randomized tie count; PITⱼ = (rⱼ + U)/(Ndraw + 1)`

**Units.** Rank is an integer from 0 through Ndraw. PIT and its empirical CDF are dimensionless; U is uniform on [0,1).

**Reading the result.** The CDF uses one selected test case per independent parent/lineage component. A diagonal-looking curve alone is not acceptance: original finite-draw KS, coverage and multiple-comparison decisions remain attached.

Implementation: `core/stream/sbi.py`, `calibrate_frozen_model`.

## Independent credible-interval coverage

`Ĉⱼ = Σg 1[θⱼ,true,g ∈ CIⱼ,g] / Nindependent; accept only if the binomial CI lies inside [q − ε, q + ε]`

**Units.** Coverage, nominal credible mass q, tolerance ε and the binomial bounds are dimensionless probabilities.

**Reading the result.** The posterior is frozen before reading held-out truth. The interval quantifies finite independent test cases; multiple draws from one parent do not increase that count. The short harmonic example fails calibration and remains disabled for deployment.

Implementation: `core/stream/sbi.py`, `calibrate_frozen_model`.

## Training-domain support

`d² = (s − μ)ᵀ C⁻¹(s − μ); foutside = count(sⱼ outside training rangeⱼ)/Nsummary`

**Units.** Summaries use their recorded normalization. Squared Mahalanobis distance and outside fraction are dimensionless.

**Reading the result.** Mean, covariance regularization and ranges come only from training observations. Both original thresholds must pass. This is a support screen, not a calibrated probability of model correctness; posterior deployment also requires accepted independent calibration.

Implementation: `core/stream/sbi.py`, `evaluate_ood`.

## Paired graph-versus-summary log-score gain

`Δg = log pgraph(θtrue,g | xg) − log psummary(θtrue,g | xg); gain probability = fraction(meanbootstrap(Δ) > declared minimum)`

**Units.** Log-score differences are dimensionless when both densities use the same recorded parameter coordinates and units.

**Reading the result.** Both frozen models must independently pass calibration on the same held-out lineage cases. Bootstrap resamples paired independent groups. A rejected gain cannot authorize graph deployment, and this conditional evaluation does not establish performance on other physical domains.

Implementation: `core/stream/sbi.py`, `graph_baseline_gain`.

## A viewing zoom is not IC refinement

The lab measures nested 100, 20 and 5 proper-kpc projected views of the same
synthetic HDF5 particles. Pixel areas and selected mass are recalculated at
each view. No particles are added. MUSIC refinement, contamination controls
and native zoom evolution are separate workflows in the main repository.


In [ ]:
if RUN is not None and (RUN / 'zoom_arrays.npz').exists():
    with np.load(RUN / 'zoom_arrays.npz') as data:
        for i in range(3):
            selected = data[f'view_{i}_selected']
            edges = data[f'view_{i}_x_edges_proper_kpc']
            pixel_area = (edges[1] - edges[0])**2
            mass = data[f'view_{i}_surface_density_msun_kpc2'].sum() * pixel_area
            print(i, 'selected particles:', selected.sum(), 'projected mass [Msun]:', mass)
    display(Image(filename=str(RUN / 'zoom.png')))


## Native MUSIC refinement

MUSIC initialized 2,744 refined gas particles, 2,744 refined DM particles and
5,265 coarse boundary particles. GADGET evolved the load and saved the epoch
shown below. Each panel crops the same projected coordinates in comoving
Mpc/h. To convert a plotted length to proper Mpc, multiply by the saved
scale factor and divide by h. The zero-transfer control exposes the
refinement geometry and tests the handoff between programs.

![Native zoom geometry](native_zoom.png)

The plotted coordinates are in `native_zoom.npz`. The snapshot hash, species
counts and view limits are in `native_zoom.json`. See
`optional_tests/zoom_reference.py` for the rendering code.


## Inspect fresh optional-library calculations

Run the `libraries` profile to produce these files locally. Each backend has
its own process log, status, numerical assertions, units and scope. The cell
verifies the published file hashes before reading arrays. A chain or trained
network is an execution artifact; its existence does not establish calibration.
The FSPS spectrum belongs to a separately calculated 1-Gyr, 1-Msun-formed SSP,
not the stellar marker particles in the synthetic HDF5 fixture.


In [ ]:
if RUN is not None and (RUN / 'libraries').is_dir():
    import hashlib
    published = {a['path']: a for a in receipt['artifacts']}
    for path in sorted((RUN / 'libraries').glob('*/receipt.json')):
        key = path.relative_to(RUN).as_posix()
        assert hashlib.sha256(path.read_bytes()).hexdigest() == published[key]['sha256']
        backend = json.loads(path.read_text())
        print(path.parent.name, backend['status'])
        print(backend.get('result', {}).get('scope', backend.get('reason', '')))
        arrays = path.parent / 'arrays.npz'
        if arrays.is_file():
            key = arrays.relative_to(RUN).as_posix()
            assert hashlib.sha256(arrays.read_bytes()).hexdigest() == published[key]['sha256']
            with np.load(arrays, allow_pickle=False) as values:
                print({name: (values[name].shape, str(values[name].dtype)) for name in values.files})
    spectrum = RUN / 'libraries/fsps/spectrum.png'
    if spectrum.is_file():
        key = spectrum.relative_to(RUN).as_posix()
        assert hashlib.sha256(spectrum.read_bytes()).hexdigest() == published[key]['sha256']
        display(Image(filename=str(spectrum)))
else:
    print('Select a completed libraries profile run to inspect its fresh backend outputs.')


## Inspect native solvers, scenes, ICs, cooling and catalogues

The `engines` profile adds actual CLASS/AxiCLASS calculations, a Blender
still scene, a fresh CLASS-to-monofonIC IC control and a 32-cell Grackle
atomic cooling control. MUSIC adds ten uniform/nested/periodic analytic-transfer
realizations and seven expected rejections, with 10,753 particles in the nested
reference. These are normalization controls, not a target-halo zoom. The profile
requires their installed native software. Rockstar and VELOCIraptor use separate 32-DM-particle synthetic
fixtures with the same seed. Their two saved epochs are repeated samples, not
an evolution. The catalogue counts may be zero; different settings and mass
definitions do not establish finder agreement or halo accuracy.
The next cell verifies worker receipts and arrays against the run artifact hashes, checks each
sealed Boltzmann bundle, and replays the dimensionless handoff:
$$\Delta^2(k,z)=\frac{k^3P_{\rm CLASS}(k,z)}{2\pi^2}.$$
Here $k$ is in $h/{\rm Mpc}$ and $P$ is in $({\rm Mpc}/h)^3$. These are linear
solver outputs, not measured particle mesh powers. The particle handoff stores
the logarithms at z=49. The Blender picture uses 32 arbitrary display points
and enlarged display glyphs. Its separate native verification reopens the
saved `.blend`; reading this cell does not launch another renderer or a movie.
The monofonIC control uses a 4³ grid per species (128 total), distinct from the
32-particle quick fixture. Its original HDF5 positions are comoving kpc/h;
stored GADGET velocities become peculiar velocities after multiplying by sqrt(a).
The cell rechecks IDs and the cosmological gas mass fraction from the arrays.
Grackle evolves 32 independent constant-density cells, not particles. Its
atomic H/He network has no UV, metals, dust or molecules. Species fractions
are mass divided by total gas mass, so HI+HII=0.76 and HeI+HeII+HeIII=0.24.
The signed instantaneous cooling time is u/(du/dt), negative for net cooling;
its magnitude is a timescale. Temperatures are native calculations in kelvin.


In [ ]:
engine_root = RUN / 'engines' if RUN is not None else None
if engine_root is None or not engine_root.is_dir():
    print('Select a completed engines-profile run to inspect native package outputs.')
else:
    from hashlib import sha256
    from core.linear import verify_spectrum_directory
    published = {row['path']: row for row in receipt['artifacts']}
    def check_published(path):
        key = path.relative_to(RUN).as_posix()
        assert sha256(path.read_bytes()).hexdigest() == published[key]['sha256'], path
    for backend in ('class', 'axiclass', 'blender', 'monofonic', 'grackle', 'music2', 'rockstar', 'velociraptor'):
        directory = engine_root / backend
        result_path = directory / 'receipt.json'
        if not result_path.is_file():
            print(backend, 'has no receipt in this run')
            continue
        check_published(result_path)
        worker = json.loads(result_path.read_text())
        print(backend, worker['status'])
        if worker['status'] != 'PASS':
            print(worker.get('reason'))
            continue
        print(worker['result']['implementation'])
        print(worker['result']['scope'])
        check_published(directory / 'arrays.npz')
        if backend in ('class', 'axiclass'):
            manifest = directory / worker['result']['bundle_manifest']
            check_published(manifest)
            bundle = verify_spectrum_directory(manifest.parent)
            handoff = np.loadtxt(manifest.parent / 'pk_gadget.txt')
            with np.load(directory / 'arrays.npz', allow_pickle=False) as data:
                k, power = data['start_k_h_mpc'], data['start_power_mpc_h_cubed']
                delta_squared = k**3 * power / (2 * np.pi**2)
                np.testing.assert_allclose(handoff[:, 0], np.log10(k), rtol=0, atol=1e-9)
                np.testing.assert_allclose(handoff[:, 1], np.log10(delta_squared), rtol=0, atol=1e-9)
                print('Original start modes:', len(k), '| units:', bundle['units'])
            figure = manifest.parent / 'matter_power.png'
        elif backend == 'monofonic':
            check_published(directory / 'inspection.json')
            check_published(directory / worker['result']['hdf5'])
            with np.load(directory / 'arrays.npz', allow_pickle=False) as data:
                ids = np.concatenate([data['ids_0'], data['ids_1']])
                assert len(ids) == len(np.unique(ids)) == 128
                for species in (0, 1):
                    assert data[f'position_{species}_kpc_h'].shape == (64, 3)
                gas = data['mass_0_1e10_msun_h'].sum()
                dm = data['mass_1_1e10_msun_h'].sum()
                np.testing.assert_allclose(gas / (gas + dm), .0224 / (.0224 + .12), rtol=0, atol=1e-6)
                print('128 IC particles; original gas mass fraction:', gas / (gas + dm))
            figure = directory / 'positions.png'
        elif backend == 'grackle':
            check_published(directory / 'cooling.csv')
            with np.load(directory / 'arrays.npz', allow_pickle=False) as data:
                assert data['temperature_K'].shape == (2, 32)
                np.testing.assert_allclose(data['HI_mass_fraction'] + data['HII_mass_fraction'], .76, rtol=0, atol=1e-10)
                helium = sum(data[key] for key in ('HeI_mass_fraction', 'HeII_mass_fraction', 'HeIII_mass_fraction'))
                np.testing.assert_allclose(helium, .24, rtol=0, atol=1e-10)
                assert (data['cooling_time_s'] < 0).all()
                assert (data['specific_energy_erg_g'][1] < data['specific_energy_erg_g'][0]).all()
                print('Native 32-cell atomic cooling; temperatures [K]:', data['temperature_K'].min(), data['temperature_K'].max())
            figure = directory / 'cooling.png'
        elif backend == 'music2':
            report_path = directory / worker['result']['report']
            check_published(report_path)
            music = json.loads(report_path.read_text())
            assert music['status'] == music['nested']['status'] == music['periodic']['status'] == 'PASS'
            assert len(music['execution']['rejections']) == 7
            assert all(row['exit_code'] != 0 for row in music['execution']['rejections'].values())
            for family in ('uniform', 'nested', 'periodic'):
                for case, inputs in music[family]['inputs'].items():
                    for filename, expected in inputs.items():
                        original = directory / 'controls' / case / filename
                        check_published(original)
                        assert sha256(original.read_bytes()).hexdigest() == expected
            import h5py
            with h5py.File(directory / 'controls/dms-reference/zoom_fixture.hdf5') as native, np.load(directory / 'arrays.npz', allow_pickle=False) as data:
                count = 0
                for species in (0, 1, 2):
                    np.testing.assert_array_equal(native[f'PartType{species}/Coordinates'][:], data[f'Coordinates_{species}'])
                    count += len(data[f'ParticleIDs_{species}'])
                assert count == worker['result']['reference_particles'] == 10753
                print('Nested reference particles:', count, '| velocity/displacement residuals:', worker['result']['velocity_displacement_relative_residuals'])
            figure = directory / 'nested.png'
        elif backend in ('rockstar', 'velociraptor'):
            check_published(directory / 'catalogue.json')
            check_published(directory / worker['result']['native_report'])
            with np.load(directory / 'arrays.npz', allow_pickle=False) as data:
                assert data['input_position_comoving_kpc_h'].shape == (32, 3)
                assert len(np.unique(data['input_ids'])) == 32
                np.testing.assert_array_equal(data['object_count'], worker['result']['object_counts'])
                catalogue_output = json.loads((directory / 'catalogue.json').read_text())
                assert len(catalogue_output['objects']) == len(data['object_mass_msun']) == data['object_count'][-1]
                print('Native object counts:', data['object_count'], '| definition:', worker['result']['mass_definition'])
            if backend == 'rockstar':
                for snapshot, count in zip(worker['result']['catalogue_snapshots'], worker['result']['object_counts']):
                    path = directory / f'catalogue-{snapshot:03}.json'
                    check_published(path)
                    assert len(json.loads(path.read_text())['objects']) == count
            else:
                report = json.loads((directory / 'velociraptor/report.json').read_text())
                check_published(directory / 'velociraptor/report.sha256')
                assert sha256((directory / 'velociraptor/report.json').read_bytes()).hexdigest() == (directory / 'velociraptor/report.sha256').read_text().strip()
                for name, expected in report['artifacts'].items():
                    path = directory / 'velociraptor' / name
                    check_published(path)
                    assert sha256(path.read_bytes()).hexdigest() == expected
            figure = directory / 'catalogue.png'
        else:
            check_published(directory / 'particles.blend')
            check_published(directory / 'verified-scene.json')
            print('Fresh Blender load:', json.loads((directory / 'verified-scene.json').read_text()))
            figure = directory / 'preview.png'
        check_published(figure)
        display(Image(filename=str(figure)))


## Native tree links and filament topology

The tree control supplies **128 synthetic catalogue objects per epoch**, at
a=0.99 and a=1. It creates no particles; the input Np=1000 is a declared toy
catalogue property. The production Consistent Trees route writes actual
hlists. Their integer IDs, descendant IDs and descendant scale factors define
the links. The next cell reads these original tables, compares the saved
arrays and checks that every link advances to its recorded later epoch.
Positions remain comoving Mpc/h and masses Msun/h. This is not a physically
evolved merger history or an accuracy benchmark.

DisPerSE uses **64 DM samples**, its production minimum, on a 16³ periodic
CIC mass mesh. With one-cell Gaussian density smoothing it processes
F=log10(1 + smoothed rho / mean rho), a dimensionless scalar. The density
smoothing scale is 0.1/16=0.00625 comoving Mpc/h. Native mse finds the ascending
persistence skeleton at cut=0.1; skelconv exports topology and display geometry.
One display-smoothing iteration does not replace the separately retained
unsmoothed topology. The XY picture is **boolean skeleton occupancy**, not
projected density or filament thickness. The cell reconstructs that occupancy
from the saved periodic segments and checks the original sealed artifacts.
Implementation: `core/halos.py:run_consistent_trees` and
`core/visualization.py:_load_disperse_skeleton/_disperse_filament_projection_map`.


In [ ]:
if engine_root is None or not engine_root.is_dir():
    print('Select a completed engines run for tree and filament controls.')
else:
    from core.analysis import rockstar_table_from_path
    import h5py
    for backend in ('consistent-trees', 'disperse'):
        directory = engine_root / backend
        receipt_path = directory / 'receipt.json'
        if not receipt_path.is_file():
            print(backend, 'has no receipt in this run')
            continue
        check_published(receipt_path)
        worker = json.loads(receipt_path.read_text())
        print(backend, worker['status'])
        if worker['status'] != 'PASS':
            print(worker.get('reason'))
            continue
        check_published(directory / 'arrays.npz')
        print(worker['result']['scope'])
        with np.load(directory / 'arrays.npz', allow_pickle=False) as data:
            if backend == 'consistent-trees':
                native = []
                for name in worker['result']['native_catalogues']:
                    path = directory / name
                    check_published(path)
                    native.append(rockstar_table_from_path(path))
                for key in ('id','scale','desc_id','desc_scale','x','y','z','mvir','orig_halo_id'):
                    np.testing.assert_array_equal(np.concatenate([table[key] for table in native]), data[key])
                assert len(np.unique(data['id'])) == 256
                links = data['link_rows']
                assert links.shape == (128, 2) and ((links >= 0) & (links < 256)).all()
                source, target = links.T
                np.testing.assert_array_equal(data['desc_id'][source], data['id'][target])
                np.testing.assert_allclose(data['desc_scale'][source], data['scale'][target], rtol=0, atol=1e-12)
                assert (data['scale'][target] > data['scale'][source]).all()
                print('Original tree:', len(data['id']), 'nodes;', len(links), 'forward links')
                figure = directory / 'trees.png'
            else:
                manifest = directory / worker['result']['manifest']
                check_published(manifest)
                contract = json.loads(manifest.read_text())
                for item in contract['assets']:
                    path = manifest.parent / item['path']
                    check_published(path)
                    assert sha256(path.read_bytes()).hexdigest() == item['sha256']
                with h5py.File(manifest.parent / 'skeleton.hdf5') as original:
                    np.testing.assert_array_equal(original['points_comoving_mpc_h'][:], data['points_comoving_mpc_h'])
                    np.testing.assert_array_equal(original['vtk_polyline_connectivity'][:], data['vtk_polyline_connectivity'])
                points, lines = data['points_comoving_mpc_h'], data['vtk_polyline_connectivity']
                occupancy = np.zeros((16,16), dtype=bool)
                cursor = 0
                while cursor < len(lines):
                    count = int(lines[cursor]); rows = lines[cursor+1:cursor+1+count]
                    cursor += count+1
                    assert count >= 2 and len(rows) == count and ((rows >= 0) & (rows < len(points))).all()
                    for start, finish in zip(points[rows[:-1]], points[rows[1:]]):
                        delta = finish-start; delta -= .1*np.round(delta/.1)
                        samples = max(2, int(np.ceil(np.max(np.abs(delta))*16/.1))+1)
                        xyz = np.mod(start + np.linspace(0,1,samples)[:,None]*delta, .1)
                        bins = np.floor(xyz[:,:2]*16/.1).astype(int) % 16
                        occupancy[bins[:,0],bins[:,1]] = True
                np.testing.assert_array_equal(occupancy, data['projection_occupancy'])
                print('Original skeleton:', len(points), 'points;', np.count_nonzero(occupancy), 'occupied XY cells')
                figure = directory / 'filaments.png'
        check_published(figure)
        display(Image(filename=str(figure)))


## Independent OpenGadget3 scattering

This optional control requires an existing Linux rare/directed reference build.
Without it the checklist records UNAVAILABLE. It evolves 32 beam particles at
three saved epochs with gravity disabled and a deliberately high toy total
cross section of 100 cm²/g. The checks use exact IDs, fixed macro masses,
P=sum(m v) and K=sum(m |v|²)/2. Momentum residuals divide by the initial
sum(m |v|); energy residuals divide by initial K. The threshold is 1e-10.
K uses Msun (km/s)² and P uses Msun km/s. Static proper elapsed seconds equal
native time multiplied by L/(V h), as in the production control reader.
Changed velocities demonstrate scattering; they are not an event count or a
calibrated cross-section measurement.

Set RESEARCH2_LAB_SIDM to a standalone completed worker directory to inspect
Linux outputs from another environment. Otherwise the cell uses this run's
OpenGadget3 worker. It verifies retained output hashes and replays conservation
without executing the native binary.


In [ ]:
sidm_specified = os.environ.get('RESEARCH2_LAB_SIDM')
sidm_directory = Path(sidm_specified).resolve() if sidm_specified else (engine_root / 'opengadget3' if engine_root is not None else None)
if sidm_directory is None or not (sidm_directory / 'receipt.json').is_file():
    print('No independent worker output selected. Set RESEARCH2_LAB_SIDM to its completed directory.')
else:
    sidm_receipt = json.loads((sidm_directory / 'receipt.json').read_text())
    print('OpenGadget3:', sidm_receipt['status'])
    if sidm_receipt['status'] != 'PASS':
        print(sidm_receipt.get('reason'))
    else:
        from hashlib import sha256
        for name, expected in sidm_receipt['result']['artifact_sha256'].items():
            path = (sidm_directory / name).resolve()
            assert path.is_relative_to(sidm_directory)
            assert sha256(path.read_bytes()).hexdigest() == expected
        with np.load(sidm_directory / 'arrays.npz', allow_pickle=False) as data:
            mass, velocity = data['mass_msun'], data['velocity_peculiar_kms']
            assert velocity.shape == (3,32,3)
            np.testing.assert_array_equal(data['particle_ids'], np.broadcast_to(data['particle_ids'][0], (3,32)))
            momentum = np.sum(mass[:,:,None] * velocity, axis=1)
            energy = .5*np.sum(mass*np.sum(velocity**2, axis=2), axis=1)
            np.testing.assert_array_equal(momentum, data['momentum_msun_kms'])
            np.testing.assert_array_equal(energy, data['kinetic_energy_msun_kms2'])
            momentum_scale = np.sum(mass[0]*np.linalg.norm(velocity[0], axis=1))
            assert np.max(np.linalg.norm(momentum-momentum[0],axis=1)/momentum_scale) < 1e-10
            assert np.max(abs(energy/energy[0]-1)) < 1e-10
            changed = np.count_nonzero(np.linalg.norm(velocity-velocity[0], axis=2)>1e-9,axis=1)
            np.testing.assert_array_equal(changed, data['changed_velocity_particles'])
            assert changed[-1] > 0
            print('Changed-velocity particles:', changed)
        display(Image(filename=str(sidm_directory / 'scattering.png')))


## Native gas, stellar births and black-hole sources

The `physics` profile adds three actual GADGET source controls to the seven
quick measurements; `complete` also includes the libraries and native engines.
Each source control uses a 4³ gas grid, **64 gas particles**. Stellar particles
are absent initially. Native SH03 conversions carry SSPBirthMode=1 and their
parent IDs, birth times and source journal are retained. The stellar seed is
the selected StellarRandomSeed, separate from continuous hydrodynamics.

The gas run shows the mass-weighted internal energy after hydrodynamics and
implicit atomic cooling. The star-forming run uses an effective multiphase
EOS, so its increasing gas energy is not a single-phase temperature estimate.
Its production history audit reconciles six-species masses, momentum and
native birth events with the frozen dimensional tolerances. Delayed stellar
returns and SPS light synthesis are separate controls.

The BH run contains one explicitly seeded 10-Msun sink. It starts from a
separately declared compressed gas IC; the original sealed grid is retained.
The toy background contains gas/BH baryonic matter. Saved BH_Mass is the
physical mass and BH_InflowRate is converted through its own cgs attributes.
Header/Time is scale factor a in this run, and physical native L/V time in the
other two. Positive mass growth/inflow is an execution check; full BH source
budget reconciliation, formation, galaxy accuracy and convergence are not
inferred from this example.

The next cell checks original HDF5 counts/energies against the saved arrays,
verifies the stellar source-history audit, and displays the original figures.
Implementation: `core/research/controls.py:prepare_native_source_control` →
`core/evolution.py:build_gadget/gadget_param_text` → native GADGET →
`core/stars.py:audit_native_hydro_history` for stellar reconciliation.


In [ ]:
physics_root = RUN / 'physics' if RUN is not None else None
if physics_root is None or not physics_root.is_dir():
    print('Select a completed physics or complete-profile run for native source controls.')
else:
    from hashlib import sha256
    from core.data import dataset_to_cgs_factor
    from core.stars import verify_native_hydro_history
    import h5py
    published = {item['path']: item for item in receipt['artifacts']}
    def check_physics_file(path):
        assert sha256(path.read_bytes()).hexdigest() == published[path.relative_to(RUN).as_posix()]['sha256']
    for backend in ('gas','stars','black-holes'):
        directory = physics_root / backend
        path = directory / 'receipt.json'
        check_physics_file(path)
        worker = json.loads(path.read_text())
        print(backend, worker['status'])
        if worker['status'] != 'PASS':
            print(worker.get('reason')); continue
        print('Recorded source matches current checkout:', all(sha256((ROOT/name).read_bytes()).hexdigest()==expected for name,expected in worker['source_sha256'].items()))
        check_physics_file(directory / 'arrays.npz')
        with np.load(directory / 'arrays.npz',allow_pickle=False) as data:
            np.testing.assert_array_equal(data['particle_counts'],worker['result']['particle_counts'])
            for epoch,snapshot in enumerate(sorted((directory / 'data').glob('snapshot_*.hdf5'))):
                check_physics_file(snapshot)
                with h5py.File(snapshot) as native:
                    np.testing.assert_array_equal(native['Header'].attrs['NumPart_Total'],data['particle_counts'][epoch])
                    actual = float(native['Header'].attrs['Time'])
                    assert abs(actual-data['native_time'][epoch])<1e-12
                    a = actual if worker['result']['comoving'] else 1.
                    h = .7 if worker['result']['comoving'] else 1.
                    gas = native['PartType0']
                    u = gas['InternalEnergy'][:]*dataset_to_cgs_factor(gas['InternalEnergy'].attrs,a,h)
                    np.testing.assert_allclose(u,data[f'gas_specific_energy_erg_g_{epoch}'],rtol=1e-14,atol=0)
                    np.testing.assert_allclose(np.average(u,weights=gas['Masses'][:]),data['gas_mass_weighted_specific_energy_erg_g'][epoch],rtol=1e-14,atol=0)
            if backend=='gas':
                assert data['gas_mass_weighted_specific_energy_erg_g'][-1]<data['gas_mass_weighted_specific_energy_erg_g'][0]
            elif backend=='stars':
                assert data['particle_counts'][0,4]==0 and data['particle_counts'][-1,4]>0
                assert (data['SSPBirthMode_2']==1).all()
                check_physics_file(directory / 'history.json')
                history = verify_native_hydro_history(directory / 'history.json')
                assert history['status']=='PASS'
                print('Native stellar history:',history['status'],'| births:',data['particle_counts'][-1,4])
            else:
                assert data['bh_physical_mass_msun'][-1]>data['bh_physical_mass_msun'][0]
                assert data['bh_inflow_msun_per_year'].max()>0
                print('BH physical mass gain [Msun]:',np.diff(data['bh_physical_mass_msun']).sum())
        check_physics_file(directory / 'physics.png')
        display(Image(filename=str(directory / 'physics.png')))


## Fresh known-input shape and density controls

Run the `figures` profile, or `complete`, to regenerate these examples from the
shipped `advanced_controls.py`. The worker creates its own synthetic HDF5 and
measures it with the production readers; it does not need the author's campaign
folders. Shapes uses 6000 particles, and the density-valley control uses 32768.
Neither is native solver evolution. The next cell verifies the worker and
parent hashes, replays the physical coordinate/mass conversion, independently
recomputes shell tensor ratios, and checks the linear derivative/covariance
identities against the original saved arrays. It reads the original images
without rerunning AGAMA or selecting a different derivative window.


In [ ]:
control_root = RUN / 'figure-controls' if RUN is not None else None
if control_root is None or not control_root.is_dir():
    print('Select a completed figures or complete profile for fresh known-input controls.')
else:
    import h5py
    from hashlib import sha256
    from core.data import dataset_to_cgs_factor
    from core.foundation import KPC_CM, MSUN_G
    published = {r['path']: r['sha256'] for r in receipt['artifacts']}
    for control in ('shapes','splashback'):
        directory = control_root / control
        path = directory / 'receipt.json'
        assert sha256(path.read_bytes()).hexdigest() == published[path.relative_to(RUN).as_posix()]
        worker = json.loads(path.read_text())
        if worker['status'] == 'UNAVAILABLE':
            print(control, 'UNAVAILABLE:', worker['reason'])
            continue
        assert worker['status'] == 'PASS'
        for name,digest in worker['artifacts_sha256'].items():
            assert sha256((directory / name).read_bytes()).hexdigest() == digest
        print(control, '| particles:',worker['result']['particles'], '| native evolution:',worker['result']['native_solver_executed'])
        with np.load(directory / 'authored-geometry.npz') as geometry:
            original = geometry['offsets_proper_kpc']
        with h5py.File(directory / 'data/snapshot_000.hdf5') as source:
            gas = source['PartType1']
            a = float(source['Header'].attrs['Time'])
            h = float(source['Parameters'].attrs['HubbleParam'])
            x = (gas['Coordinates'][:] - 50) * dataset_to_cgs_factor(gas['Coordinates'].attrs,a,h) / KPC_CM
            mass = gas['Masses'][:] * dataset_to_cgs_factor(gas['Masses'].attrs,a,h) / MSUN_G
        np.testing.assert_allclose(x,original,rtol=1e-12,atol=1e-12)
        np.testing.assert_allclose(mass,1.,rtol=1e-12,atol=0)
        structure = json.loads((directory / 'structure.json').read_text())
        if control == 'shapes':
            for shell in structure['density_shapes']['dm']['shells']:
                metric = np.asarray(shell['selection_metric'])
                r2 = np.einsum('ni,ij,nj->n',x,metric,x)
                keep = (r2 >= shell['lower_radius_kpc']**2) & (r2 < shell['upper_radius_kpc']**2)
                assert keep.sum() == shell['particle_count']
                tensor = np.einsum('n,ni,nj->ij',mass[keep],x[keep],x[keep]) / mass[keep].sum()
                eigen = np.linalg.eigvalsh(tensor)[::-1]
                ratios = np.sqrt(eigen[1:] / eigen[0])
                np.testing.assert_allclose(ratios,[shell['b_over_a'],shell['c_over_a']],rtol=0,atol=1e-4)
            images = sorted(directory.glob('figures/gallery/figures/png/structure/*axis_orientation_twist.png'))
            print('Original ellipsoidal-window tensor ratios replayed.')
        else:
            with np.load(directory / 'structure.json.data/arrays.npz') as raw:
                rho = raw['profile_dm_density_msun_kpc3']
                error = raw['profile_dm_density_shot_noise_msun_kpc3']
            with np.load(directory / 'environment.json.data/arrays.npz') as raw:
                prefix = 'smoothing_00_halo_00_splashback_'
                operator = raw[prefix+'derivative_operator']
                support = raw[prefix+'supported']
                logarithm = np.zeros_like(rho); logarithm[rho>0] = np.log(rho[rho>0])
                np.testing.assert_allclose((operator @ logarithm)[support],raw[prefix+'logarithmic_slope'][support],rtol=1e-10,atol=1e-10)
                covariance = np.diag(np.divide(error,rho,out=np.zeros_like(rho),where=rho>0)**2)
                np.testing.assert_allclose(operator @ covariance @ operator.T,raw[prefix+'slope_covariance'],rtol=1e-10,atol=1e-10)
            images = sorted(directory.glob('figures/gallery/figures/png/structure/*splashback.png'))
            print('Original derivative and covariance replayed; feature:',worker['result']['measurements']['feature']['status'])
        assert images
        display(Image(filename=str(images[0])))


## Replay the portable analytic flyby

The shipped `orbits` control creates 32 particles in two authored apertures at
three epochs. It calls the production graph, periodic-image, clock and Hermite
trajectory/event pipeline. This is a known straight-line geometry, not evolved
halos, finder detections or certified physical ancestry. Its host boundary is
a fixed 1.5-kpc test aperture, not a measured R200. The next cell derives the
analytic infall/pericentre times from the original saved relative states and
boundary, checks the reported candidates and displays the original figure.
The interpolation-uncertified status and saved brackets remain explicit.


In [ ]:
orbit_directory = RUN / 'figure-controls/orbits' if RUN is not None else None
if orbit_directory is None or not (orbit_directory / 'receipt.json').is_file():
    print('Select a completed figures or complete profile with the portable orbits control.')
else:
    from hashlib import sha256
    import h5py
    from core.foundation import KPC_CM, SECONDS_PER_YEAR
    worker_path = orbit_directory / 'receipt.json'
    published = {r['path']: r['sha256'] for r in receipt['artifacts']}
    assert sha256(worker_path.read_bytes()).hexdigest() == published[worker_path.relative_to(RUN).as_posix()]
    worker = json.loads(worker_path.read_text())
    if worker['status'] == 'UNAVAILABLE':
        print('orbits UNAVAILABLE:',worker['reason'])
    else:
        assert worker['status'] == 'PASS' and not worker['result']['native_solver_executed']
        assert not worker['result']['trajectory_certified']
        for name,digest in worker['artifacts_sha256'].items():
            assert sha256((orbit_directory / name).read_bytes()).hexdigest() == digest
        with np.load(orbit_directory / 'events.json.data/arrays.npz') as data:
            time = data['times_myr']
            separation = data['object_position_proper_kpc'] - data['host_position_proper_kpc']
            relative_derivative = data['object_velocity_derivative_kms'] - data['host_velocity_derivative_kms']
            assert data['object_valid_intervals'].all() and data['host_valid_intervals'].all()
            speed = (separation[-1,0]-separation[0,0])/(time[-1]-time[0])
            velocity_kpc_myr = relative_derivative * 1e5 * (1e6*SECONDS_PER_YEAR) / KPC_CM
            np.testing.assert_allclose(velocity_kpc_myr,np.tile([speed,0.,0.],(3,1)),rtol=1e-10,atol=1e-12)
            t_peri = time[0] - separation[0,0]/speed
            impact = np.linalg.norm(separation[:,1:],axis=1)
            np.testing.assert_allclose(impact,impact[0],rtol=0,atol=1e-12)
            with h5py.File(orbit_directory / 'data/fof_subhalo_tab_000.hdf5') as source:
                boundary = source['Group/Group_R_Crit200'][1] * source['Header'].attrs['Time'] / source['Parameters'].attrs['HubbleParam']
            t_infall = t_peri - np.sqrt(boundary**2-impact[0]**2)/speed
            for code,expected in ((0,t_peri),(1,t_infall)):
                values = data['event_time_myr'][data['event_kind_code']==code]
                assert len(values)==1
                np.testing.assert_allclose(values,[expected],rtol=0,atol=worker['result']['time_absolute_tolerance_myr'])
            assert len(data['merger_time_myr'])==0
        print('Derived infall/pericentre [Myr]:',t_infall,t_peri,'| native evolution: False | interpolation: NOT_CERTIFIED')
        image = next(orbit_directory.glob('figures/gallery/figures/png/assembly/*assembly-events.png'))
        display(Image(filename=str(image)))


## Read and replay the native GADGET control

Select a `native` or `combined` run to execute this cell. The isolated
self-gravity control uses native units with $G=1$ and a length/velocity time
unit; its output clock is not a cosmological scale factor or a number of Myr.
For the saved masses, velocities and self-gravity potentials,
$$K=\frac12\sum_i m_i|v_i|^2,\quad
W=\frac12\sum_i m_i\Phi_i,\quad E=K+W,\quad
L=\sum_i m_i(x_i\times v_i).$$
The factor one half in $W$ prevents counting each pair twice. This equation
applies to this isolated self-gravity case. A prescribed external potential
would have a different potential-energy accounting rule. The cell recomputes
the original energy and angular-momentum residuals, checks their numerical
archives and displays the measured energy drift. The saved thresholds and
reference floors remain visible; a tiny short run does not establish general
force accuracy or physical convergence.
Implementation: `optional_tests/lab.py:native_control` →
`core/research/controls.py:run_control_campaign` → native GADGET →
`core/research/controls.py:_analyze_case` → saved report and NPZ.


In [ ]:
native_path = RUN / 'native-control/plummer-tree/analysis.json' if RUN is not None else None
if native_path is None or not native_path.is_file():
    print('Select a completed native or combined run to replay actual native diagnostics.')
else:
    from hashlib import sha256
    from core.data import verify_numeric_audit
    native = verify_numeric_audit(native_path)
    published = {row['path']: row for row in receipt['artifacts']}
    arrays_path = native_path.parent / native['numeric_data']['directory'] / native['numeric_data']['array_contract']['path']
    for path in (native_path, arrays_path):
        key = path.relative_to(RUN).as_posix()
        assert sha256(path.read_bytes()).hexdigest() == published[key]['sha256']
    print('Clock:', native['clock'], '| numerical status:', native['status'])
    for finding in native['findings']:
        print(finding['metric'], finding['status'], 'threshold', finding.get('threshold'))
    with np.load(arrays_path, allow_pickle=False) as data:
        times = data['actual_times']
        mass, position, velocity, potential = [data[key] for key in ('mass', 'position', 'velocity', 'potential')]
        kinetic = .5 * np.sum(mass * np.sum(velocity**2, axis=-1), axis=1)
        binding = .5 * np.sum(mass * potential, axis=1)
        energy = kinetic + binding
        angular = np.sum(mass[..., None] * np.cross(position, velocity), axis=1)
        energy_drift = energy - energy[0]
        np.testing.assert_allclose(energy_drift, data['isolated_system_energy_residual'], rtol=1e-12, atol=1e-15)
        np.testing.assert_allclose(angular - angular[0], data['isolated_system_angular_residual'], rtol=1e-12, atol=1e-15)
        print('Actual times:', times, '| particles per epoch:', mass.shape[1])
        print('K, W, E in native M V^2:', np.column_stack((kinetic, binding, energy)))
        fig, ax = plt.subplots(figsize=(7, 4), layout='constrained')
        ax.plot(times, energy_drift, 'o-', color='#126b73')
        ax.set(xlabel='Actual isolated time [native L/V]', ylabel='E(t) - E(0) [native M V²]',
               title='Native GADGET energy drift · original saved values')
        ax.grid(alpha=.2)
        display(fig)
        plt.close(fig)


## All registered recipes: provenance before interpretation

Every entry below is generated from `configs/research/graph_producers.json`.
Archived PNGs show actual retained products; their NPZ files contain the plotted
numerical sidecars. **No retained example** means execution is still unproven.
A zero-support example proves rendering with masks, not a supported measurement.
The portable reference contracts retain original workstation paths for lineage;
those original native inputs are not all part of this optional download.


### cosmology / input_power

Follow the CLASS bundle, native estimator policy, exact saved epoch, k units, support masks and covariance. A ratio is not a new simulation.

Required contract: `Validated schema2 linear bundle; current sealed native analysis; matched exact-epoch/grid products plus source-bound full joint covariance and actual IC phase verification`

```json
{
  "family": "cosmology",
  "variant": "input_power",
  "producer_route": [
    {
      "module": "core.power",
      "function": "export_linear_power"
    }
  ],
  "input_contract": "Validated schema2 linear bundle; current sealed native analysis; matched exact-epoch/grid products plus source-bound full joint covariance and actual IC phase verification",
  "required_measurement": "Input species/total spectra and normalization",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.power.export_linear_power",
      "path": "core/power.py",
      "line": 76,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 122 supported plotted entries.

![Original plot](cosmology__input_power.png)

Arrays: `cosmology__input_power.npz`; figure contract: `cosmology__input_power.json`.

- k [1 / Mpc] versus input_power [Mpc^3]


### cosmology / evolved_power

Follow the CLASS bundle, native estimator policy, exact saved epoch, k units, support masks and covariance. A ratio is not a new simulation.

Required contract: `Validated schema2 linear bundle; current sealed native analysis; matched exact-epoch/grid products plus source-bound full joint covariance and actual IC phase verification`

```json
{
  "family": "cosmology",
  "variant": "evolved_power",
  "producer_route": [
    {
      "module": "core.power",
      "function": "export_evolved_power"
    }
  ],
  "input_contract": "Validated schema2 linear bundle; current sealed native analysis; matched exact-epoch/grid products plus source-bound full joint covariance and actual IC phase verification",
  "required_measurement": "Saved-epoch species/total spectra with assignment correction",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.power.export_evolved_power",
      "path": "core/power.py",
      "line": 187,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 6 supported plotted entries.

![Original plot](cosmology__evolved_power.png)

Arrays: `cosmology__evolved_power.npz`; figure contract: `cosmology__evolved_power.json`.

- k [1 / Mpc] versus evolved_power [Mpc^3]


### cosmology / power_ratios

Follow the CLASS bundle, native estimator policy, exact saved epoch, k units, support masks and covariance. A ratio is not a new simulation.

Required contract: `Validated schema2 linear bundle; current sealed native analysis; matched exact-epoch/grid products plus source-bound full joint covariance and actual IC phase verification`

```json
{
  "family": "cosmology",
  "variant": "power_ratios",
  "producer_route": [
    {
      "module": "core.power",
      "function": "export_power_ensemble"
    }
  ],
  "input_contract": "Validated schema2 linear bundle; current sealed native analysis; matched exact-epoch/grid products plus source-bound full joint covariance and actual IC phase verification",
  "required_measurement": "Matched epoch, cosmology and bin masks for spectrum ratios",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.power.export_power_ensemble",
      "path": "core/power.py",
      "line": 443,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 2 supported plotted entries.

![Original plot](cosmology__power_ratios.png)

Arrays: `cosmology__power_ratios.npz`; figure contract: `cosmology__power_ratios.json`.

- k [1 / Mpc] versus power_ratios [dimensionless]


### cosmology / transfer_comparison

Follow the CLASS bundle, native estimator policy, exact saved epoch, k units, support masks and covariance. A ratio is not a new simulation.

Required contract: `Validated schema2 linear bundle; current sealed native analysis; matched exact-epoch/grid products plus source-bound full joint covariance and actual IC phase verification`

```json
{
  "family": "cosmology",
  "variant": "transfer_comparison",
  "producer_route": [
    {
      "module": "core.power",
      "function": "export_linear_power"
    }
  ],
  "input_contract": "Validated schema2 linear bundle; current sealed native analysis; matched exact-epoch/grid products plus source-bound full joint covariance and actual IC phase verification",
  "required_measurement": "Signed density and velocity transfer columns on identical k grid",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.power.export_linear_power",
      "path": "core/power.py",
      "line": 76,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 610 supported plotted entries.

![Original plot](cosmology__transfer_comparison.png)

Arrays: `cosmology__transfer_comparison.npz`; figure contract: `cosmology__transfer_comparison.json`.

- k [1 / Mpc] versus transfer_comparison [dimensionless]
- k [1 / Mpc] versus transfer_comparison [Mpc^-1]


### cosmology / matched_residuals

Follow the CLASS bundle, native estimator policy, exact saved epoch, k units, support masks and covariance. A ratio is not a new simulation.

Required contract: `Validated schema2 linear bundle; current sealed native analysis; matched exact-epoch/grid products plus source-bound full joint covariance and actual IC phase verification`

```json
{
  "family": "cosmology",
  "variant": "matched_residuals",
  "producer_route": [
    {
      "module": "core.power",
      "function": "export_power_ensemble"
    }
  ],
  "input_contract": "Validated schema2 linear bundle; current sealed native analysis; matched exact-epoch/grid products plus source-bound full joint covariance and actual IC phase verification",
  "required_measurement": "Matched phases, units and residual covariance",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.power.export_power_ensemble",
      "path": "core/power.py",
      "line": 443,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 2 supported plotted entries.

![Original plot](cosmology__matched_residuals.png)

Arrays: `cosmology__matched_residuals.npz`; figure contract: `cosmology__matched_residuals.json`.

- k [1 / Mpc] versus matched_residuals [Mpc^3]


### abundance / differential_hmf

Counts are normalized by the declared volume and log-bin widths. Particle thresholds do not establish physical completeness.

Required contract: `Sealed analysis_products.h5 with snap_*/pk, hmf, profiles or main_progenitor_histories/data/histories/*/track; dataset units/quantity/epoch/provenance declarations required`

```json
{
  "family": "abundance",
  "variant": "differential_hmf",
  "producer_route": [
    {
      "module": "core.figures",
      "function": "export_native_analysis_figures"
    }
  ],
  "input_contract": "Sealed analysis_products.h5 with snap_*/pk, hmf, profiles or main_progenitor_histories/data/histories/*/track; dataset units/quantity/epoch/provenance declarations required",
  "required_measurement": "Resolved halo counts, volume and log-mass bins",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.figures.export_native_analysis_figures",
      "path": "core/figures.py",
      "line": 2138,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 1 supported plotted entries.

![Original plot](abundance__differential_hmf.png)

Arrays: `abundance__differential_hmf.npz`; figure contract: `abundance__differential_hmf.json`.

- Mass [Msun/h] versus differential hmf [(Mpc/h)^-3]


### abundance / cumulative_hmf

Counts are normalized by the declared volume and log-bin widths. Particle thresholds do not establish physical completeness.

Required contract: `Sealed analysis_products.h5 with snap_*/pk, hmf, profiles or main_progenitor_histories/data/histories/*/track; dataset units/quantity/epoch/provenance declarations required`

```json
{
  "family": "abundance",
  "variant": "cumulative_hmf",
  "producer_route": [
    {
      "module": "core.figures",
      "function": "export_native_analysis_figures"
    }
  ],
  "input_contract": "Sealed analysis_products.h5 with snap_*/pk, hmf, profiles or main_progenitor_histories/data/histories/*/track; dataset units/quantity/epoch/provenance declarations required",
  "required_measurement": "Same catalogue, cumulative thresholds and volume",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.figures.export_native_analysis_figures",
      "path": "core/figures.py",
      "line": 2138,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 2 supported plotted entries.

![Original plot](abundance__cumulative_hmf.png)

Arrays: `abundance__cumulative_hmf.npz`; figure contract: `abundance__cumulative_hmf.json`.

- Mass [Msun/h] versus cumulative hmf [(Mpc/h)^-3]


### abundance / hmf_ratio

Counts are normalized by the declared volume and log-bin widths. Particle thresholds do not establish physical completeness.

Required contract: `Two current native host census products; same physical epoch/h/volume/mass definition/cut, actual IC phase PASS, shared declared physical bins and existing paired jackknife`

```json
{
  "family": "abundance",
  "variant": "hmf_ratio",
  "producer_route": [
    {
      "module": "core.flagship_abundance",
      "function": "export_matched_host_abundance"
    }
  ],
  "input_contract": "Two current native host census products; same physical epoch/h/volume/mass definition/cut, actual IC phase PASS, shared declared physical bins and existing paired jackknife",
  "required_measurement": "Matched catalogues, common mass definition and completeness",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_abundance.export_matched_host_abundance",
      "path": "core/flagship_abundance.py",
      "line": 312,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 3 supported plotted entries.

![Original plot](abundance__hmf_ratio.png)

Arrays: `abundance__hmf_ratio.npz`; figure contract: `abundance__hmf_ratio.json`.

- Mass [Msun] versus B/A [dimensionless]


### abundance / reliability_limits

Counts are normalized by the declared volume and log-bin widths. Particle thresholds do not establish physical completeness.

Required contract: `Actual complete native group publication and configured mass/count selection, source-bound IC/periodic full-parent volume and actual saved epoch`

```json
{
  "family": "abundance",
  "variant": "reliability_limits",
  "producer_route": [
    {
      "module": "core.flagship_abundance",
      "function": "export_native_host_abundance"
    }
  ],
  "input_contract": "Actual complete native group publication and configured mass/count selection, source-bound IC/periodic full-parent volume and actual saved epoch",
  "required_measurement": "Particle-count, contamination and finder completeness cuts",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_abundance.export_native_host_abundance",
      "path": "core/flagship_abundance.py",
      "line": 201,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 8 supported plotted entries.

![Original plot](abundance__reliability_limits.png)

Arrays: `abundance__reliability_limits.npz`; figure contract: `abundance__reliability_limits.json`.

- Native fof mass [Msun] versus Group particles [particle count]


### abundance / stellar_mass_function

Counts are normalized by the declared volume and log-bin widths. Particle thresholds do not establish physical completeness.

Required contract: `Pinned native galaxy/SUBFIND exports with same actual epoch/namespace, original quality/count masks and verified full-parent volume`

```json
{
  "family": "abundance",
  "variant": "stellar_mass_function",
  "producer_route": [
    {
      "module": "core.flagship_abundance",
      "function": "export_native_abundance"
    }
  ],
  "input_contract": "Pinned native galaxy/SUBFIND exports with same actual epoch/namespace, original quality/count masks and verified full-parent volume",
  "required_measurement": "Native stellar masses grouped by resolved galaxy",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_abundance.export_native_abundance",
      "path": "core/flagship_abundance.py",
      "line": 122,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 24 supported plotted entries.

![Original plot](abundance__stellar_mass_function.png)

Arrays: `abundance__stellar_mass_function.npz`; figure contract: `abundance__stellar_mass_function.json`.

- stellar_mass_function [Msun] versus Number density [count / comoving Mpc^3 / dex]


### abundance / vmax_function

Counts are normalized by the declared volume and log-bin widths. Particle thresholds do not establish physical completeness.

Required contract: `Pinned native galaxy/SUBFIND exports with same actual epoch/namespace, original quality/count masks and verified full-parent volume`

```json
{
  "family": "abundance",
  "variant": "vmax_function",
  "producer_route": [
    {
      "module": "core.flagship_abundance",
      "function": "export_native_abundance"
    }
  ],
  "input_contract": "Pinned native galaxy/SUBFIND exports with same actual epoch/namespace, original quality/count masks and verified full-parent volume",
  "required_measurement": "Resolved Vmax catalogue and common completeness",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_abundance.export_native_abundance",
      "path": "core/flagship_abundance.py",
      "line": 122,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 24 supported plotted entries.

![Original plot](abundance__vmax_function.png)

Arrays: `abundance__vmax_function.npz`; figure contract: `abundance__vmax_function.json`.

- vmax_function [km/s] versus Number density [count / comoving Mpc^3 / dex]


### abundance / rare_high_z_abundance

Counts are normalized by the declared volume and log-bin widths. Particle thresholds do not establish physical completeness.

Required contract: `Actual complete native group publication and configured mass/count selection, source-bound IC/periodic full-parent volume and actual saved epoch`

```json
{
  "family": "abundance",
  "variant": "rare_high_z_abundance",
  "producer_route": [
    {
      "module": "core.flagship_abundance",
      "function": "export_native_host_abundance"
    }
  ],
  "input_contract": "Actual complete native group publication and configured mass/count selection, source-bound IC/periodic full-parent volume and actual saved epoch",
  "required_measurement": "Actual high-redshift outputs, volume and counting uncertainty",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_abundance.export_native_host_abundance",
      "path": "core/flagship_abundance.py",
      "line": 201,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 9 supported plotted entries.

![Original plot](abundance__rare_high_z_abundance.png)

Arrays: `abundance__rare_high_z_abundance.npz`; figure contract: `abundance__rare_high_z_abundance.json`.

- Native fof mass [Msun] versus n(>M) [count / comoving Mpc^3]


### structure / density_profile

Inspect radius, density/mass units, component selection, fit residuals and shell support. A fitted parameter is conditional on its model.

Required contract: `Verified native-halo-structure-v1; original component profiles, density/equipotential shape grids, sampling masks and conditional NFW fit definitions`

```json
{
  "family": "structure",
  "variant": "density_profile",
  "producer_route": [
    {
      "module": "core.halo_products",
      "function": "export_native_structure_figures"
    }
  ],
  "input_contract": "Verified native-halo-structure-v1; original component profiles, density/equipotential shape grids, sampling masks and conditional NFW fit definitions",
  "required_measurement": "Bound membership, center and radial density bins",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.halo_products.export_native_structure_figures",
      "path": "core/halo_products.py",
      "line": 17,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 12 supported plotted entries.

![Original plot](structure__density_profile.png)

Arrays: `structure__density_profile.npz`; figure contract: `structure__density_profile.json`.

- Radius [proper kpc] versus density_profile [Msun/(proper kpc)^3]


### structure / circular_velocity

Inspect radius, density/mass units, component selection, fit residuals and shell support. A fitted parameter is conditional on its model.

Required contract: `Verified native-halo-structure-v1; original component profiles, density/equipotential shape grids, sampling masks and conditional NFW fit definitions`

```json
{
  "family": "structure",
  "variant": "circular_velocity",
  "producer_route": [
    {
      "module": "core.halo_products",
      "function": "export_native_structure_figures"
    }
  ],
  "input_contract": "Verified native-halo-structure-v1; original component profiles, density/equipotential shape grids, sampling masks and conditional NFW fit definitions",
  "required_measurement": "Enclosed total/species mass and proper radii",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.halo_products.export_native_structure_figures",
      "path": "core/halo_products.py",
      "line": 17,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 15 supported plotted entries.

![Original plot](structure__circular_velocity.png)

Arrays: `structure__circular_velocity.npz`; figure contract: `structure__circular_velocity.json`.

- Radius [proper kpc] versus circular_velocity [km/s]


### structure / velocity_dispersion

Inspect radius, density/mass units, component selection, fit residuals and shell support. A fitted parameter is conditional on its model.

Required contract: `Verified native-halo-structure-v1; original component profiles, density/equipotential shape grids, sampling masks and conditional NFW fit definitions`

```json
{
  "family": "structure",
  "variant": "velocity_dispersion",
  "producer_route": [
    {
      "module": "core.halo_products",
      "function": "export_native_structure_figures"
    }
  ],
  "input_contract": "Verified native-halo-structure-v1; original component profiles, density/equipotential shape grids, sampling masks and conditional NFW fit definitions",
  "required_measurement": "Frame-subtracted radial velocity moments and support",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.halo_products.export_native_structure_figures",
      "path": "core/halo_products.py",
      "line": 17,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 9 supported plotted entries.

![Original plot](structure__velocity_dispersion.png)

Arrays: `structure__velocity_dispersion.npz`; figure contract: `structure__velocity_dispersion.json`.

- Radius [proper kpc] versus velocity_dispersion [km/s]


### structure / anisotropy

Inspect radius, density/mass units, component selection, fit residuals and shell support. A fitted parameter is conditional on its model.

Required contract: `Verified native-halo-structure-v1; original component profiles, density/equipotential shape grids, sampling masks and conditional NFW fit definitions`

```json
{
  "family": "structure",
  "variant": "anisotropy",
  "producer_route": [
    {
      "module": "core.halo_products",
      "function": "export_native_structure_figures"
    }
  ],
  "input_contract": "Verified native-halo-structure-v1; original component profiles, density/equipotential shape grids, sampling masks and conditional NFW fit definitions",
  "required_measurement": "Radial/tangential velocity moments with finite-count errors",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.halo_products.export_native_structure_figures",
      "path": "core/halo_products.py",
      "line": 17,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 9 supported plotted entries.

![Original plot](structure__anisotropy.png)

Arrays: `structure__anisotropy.npz`; figure contract: `structure__anisotropy.json`.

- Radius [proper kpc] versus anisotropy [dimensionless]


### structure / concentration_mass

Inspect radius, density/mass units, component selection, fit residuals and shell support. A fitted parameter is conditional on its model.

Required contract: `Verified native-halo-structure-v1; original component profiles, density/equipotential shape grids, sampling masks and conditional NFW fit definitions`

```json
{
  "family": "structure",
  "variant": "concentration_mass",
  "producer_route": [
    {
      "module": "core.halo_products",
      "function": "export_native_structure_figures"
    }
  ],
  "input_contract": "Verified native-halo-structure-v1; original component profiles, density/equipotential shape grids, sampling masks and conditional NFW fit definitions",
  "required_measurement": "Resolved profile fits, mass definition and fit covariance",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.halo_products.export_native_structure_figures",
      "path": "core/halo_products.py",
      "line": 17,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 4 supported plotted entries.

![Original plot](structure__concentration_mass.png)

Arrays: `structure__concentration_mass.npz`; figure contract: `structure__concentration_mass.json`.

- Total Smoke-only declared overdensity; synthetic orbit fixture
mass [Msun] versus Concentration [dimensionless]


### structure / radial_axis_ratios

Inspect radius, density/mass units, component selection, fit residuals and shell support. A fitted parameter is conditional on its model.

Required contract: `Verified native-halo-structure-v1; original component profiles, density/equipotential shape grids, sampling masks and conditional NFW fit definitions`

```json
{
  "family": "structure",
  "variant": "radial_axis_ratios",
  "producer_route": [
    {
      "module": "core.halo_products",
      "function": "export_native_structure_figures"
    }
  ],
  "input_contract": "Verified native-halo-structure-v1; original component profiles, density/equipotential shape grids, sampling masks and conditional NFW fit definitions",
  "required_measurement": "Iterative radial inertia tensors and sampling errors",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.halo_products.export_native_structure_figures",
      "path": "core/halo_products.py",
      "line": 17,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 48 supported plotted entries.

![Original plot](structure__radial_axis_ratios.png)

Arrays: `structure__radial_axis_ratios.npz`; figure contract: `structure__radial_axis_ratios.json`.

- Proper radius [kpc] versus b_over_a [dimensionless]
- Proper radius [kpc] versus c_over_a [dimensionless]


### structure / triaxiality

Inspect radius, density/mass units, component selection, fit residuals and shell support. A fitted parameter is conditional on its model.

Required contract: `Verified native-halo-structure-v1; original component profiles, density/equipotential shape grids, sampling masks and conditional NFW fit definitions`

```json
{
  "family": "structure",
  "variant": "triaxiality",
  "producer_route": [
    {
      "module": "core.halo_products",
      "function": "export_native_structure_figures"
    }
  ],
  "input_contract": "Verified native-halo-structure-v1; original component profiles, density/equipotential shape grids, sampling masks and conditional NFW fit definitions",
  "required_measurement": "Ordered shape eigenvalues and degeneracy masks",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.halo_products.export_native_structure_figures",
      "path": "core/halo_products.py",
      "line": 17,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 8 supported plotted entries.

![Original plot](structure__triaxiality.png)

Arrays: `structure__triaxiality.npz`; figure contract: `structure__triaxiality.json`.

- Proper radius [kpc] versus Density triaxiality [dimensionless]


### structure / axis_orientation_twist

Inspect radius, density/mass units, component selection, fit residuals and shell support. A fitted parameter is conditional on its model.

Required contract: `Verified native-halo-structure-v1; original component profiles, density/equipotential shape grids, sampling masks and conditional NFW fit definitions`

```json
{
  "family": "structure",
  "variant": "axis_orientation_twist",
  "producer_route": [
    {
      "module": "core.halo_products",
      "function": "export_native_structure_figures"
    }
  ],
  "input_contract": "Verified native-halo-structure-v1; original component profiles, density/equipotential shape grids, sampling masks and conditional NFW fit definitions",
  "required_measurement": "Eigenvector continuity, sign convention and degeneracy masks",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.halo_products.export_native_structure_figures",
      "path": "core/halo_products.py",
      "line": 17,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 9 supported plotted entries.

![Original plot](structure__axis_orientation_twist.png)

Arrays: `structure__axis_orientation_twist.npz`; figure contract: `structure__axis_orientation_twist.json`.

- Proper radius [kpc] versus Adjacent unsigned axis twist [deg]


### structure / density_potential_shape

Inspect radius, density/mass units, component selection, fit residuals and shell support. A fitted parameter is conditional on its model.

Required contract: `Verified native-halo-structure-v1; original component profiles, density/equipotential shape grids, sampling masks and conditional NFW fit definitions`

```json
{
  "family": "structure",
  "variant": "density_potential_shape",
  "producer_route": [
    {
      "module": "core.halo_products",
      "function": "export_native_structure_figures"
    }
  ],
  "input_contract": "Verified native-halo-structure-v1; original component profiles, density/equipotential shape grids, sampling masks and conditional NFW fit definitions",
  "required_measurement": "Density shape and independently measured potential shape",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.halo_products.export_native_structure_figures",
      "path": "core/halo_products.py",
      "line": 17,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 18 supported plotted entries.

![Original plot](structure__density_potential_shape.png)

Arrays: `structure__density_potential_shape.npz`; figure contract: `structure__density_potential_shape.json`.

- Proper radius [kpc] versus Density and total-field shape [dimensionless]


### structure / splashback_slope_radius_depth_width

Inspect radius, density/mass units, component selection, fit residuals and shell support. A fitted parameter is conditional on its model.

Required contract: `Verified native-environment-v1 plus exactly bound host structure, original slope covariance and feature status`

```json
{
  "family": "structure",
  "variant": "splashback_slope_radius_depth_width",
  "producer_route": [
    {
      "module": "core.halo_products",
      "function": "export_native_splashback_figures"
    }
  ],
  "input_contract": "Verified native-environment-v1 plus exactly bound host structure, original slope covariance and feature status",
  "required_measurement": "Resolved outer radial slopes, fit support and feature uncertainty",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.halo_products.export_native_splashback_figures",
      "path": "core/halo_products.py",
      "line": 235,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 35 supported plotted entries.

![Original plot](structure__splashback_slope_radius_depth_width.png)

Arrays: `structure__splashback_slope_radius_depth_width.npz`; figure contract: `structure__splashback_slope_radius_depth_width.json`.

- Proper radius [kpc] versus Native logarithmic density slope [dimensionless]
- Proper radius [kpc] versus Supported valley prominence [dimensionless]
- Proper radius [kpc] versus Supported half-prominence width in ln r
[dimensionless]


### assembly / mass_scale_factor

Use the same identity graph across saved epochs. Cadence limits derivative, infall and merger timing.

Required contract: `Sealed halo-main-branch-v1 stage plus exact current identity graph; actual epoch and source binding, exact IDs, original quality and cadence masks`

```json
{
  "family": "assembly",
  "variant": "mass_scale_factor",
  "producer_route": [
    {
      "module": "core.assembly",
      "function": "export_native_assembly_figures"
    }
  ],
  "input_contract": "Sealed halo-main-branch-v1 stage plus exact current identity graph; actual epoch and source binding, exact IDs, original quality and cadence masks",
  "required_measurement": "Exact main-progenitor IDs, scale factor and consistent mass",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.assembly.export_native_assembly_figures",
      "path": "core/assembly.py",
      "line": 306,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 8 supported plotted entries.

![Original plot](assembly__mass_scale_factor.png)

Arrays: `assembly__mass_scale_factor.npz`; figure contract: `assembly__mass_scale_factor.json`.

- Actual scale factor a versus Native branch mass [Msun]


### assembly / log_mass_accretion

Use the same identity graph across saved epochs. Cadence limits derivative, infall and merger timing.

Required contract: `Sealed halo-main-branch-v1 stage plus exact current identity graph; actual epoch and source binding, exact IDs, original quality and cadence masks`

```json
{
  "family": "assembly",
  "variant": "log_mass_accretion",
  "producer_route": [
    {
      "module": "core.assembly",
      "function": "export_native_assembly_figures"
    }
  ],
  "input_contract": "Sealed halo-main-branch-v1 stage plus exact current identity graph; actual epoch and source binding, exact IDs, original quality and cadence masks",
  "required_measurement": "Time derivatives on supported mass histories",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.assembly.export_native_assembly_figures",
      "path": "core/assembly.py",
      "line": 306,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 7 supported plotted entries.

![Original plot](assembly__log_mass_accretion.png)

Arrays: `assembly__log_mass_accretion.npz`; figure contract: `assembly__log_mass_accretion.json`.

- Actual scale factor a versus Interval d ln M / d ln a [dimensionless]


### assembly / fast_slow_transition

Use the same identity graph across saved epochs. Cadence limits derivative, infall and merger timing.

Required contract: `Sealed halo-main-branch-v1 stage plus exact current identity graph; actual epoch and source binding, exact IDs, original quality and cadence masks`

```json
{
  "family": "assembly",
  "variant": "fast_slow_transition",
  "producer_route": [
    {
      "module": "core.assembly",
      "function": "export_native_assembly_figures"
    }
  ],
  "input_contract": "Sealed halo-main-branch-v1 stage plus exact current identity graph; actual epoch and source binding, exact IDs, original quality and cadence masks",
  "required_measurement": "Measured accretion transition and fit uncertainty",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.assembly.export_native_assembly_figures",
      "path": "core/assembly.py",
      "line": 306,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 8 supported plotted entries.

![Original plot](assembly__fast_slow_transition.png)

Arrays: `assembly__fast_slow_transition.npz`; figure contract: `assembly__fast_slow_transition.json`.

- Actual scale factor a versus Native mass and conditional transition [Msun]


### assembly / spin_history

Use the same identity graph across saved epochs. Cadence limits derivative, infall and merger timing.

Required contract: `Sealed halo-main-branch-v1 stage plus exact current identity graph; actual epoch and source binding, exact IDs, original quality and cadence masks`

```json
{
  "family": "assembly",
  "variant": "spin_history",
  "producer_route": [
    {
      "module": "core.assembly",
      "function": "export_native_assembly_figures"
    }
  ],
  "input_contract": "Sealed halo-main-branch-v1 stage plus exact current identity graph; actual epoch and source binding, exact IDs, original quality and cadence masks",
  "required_measurement": "Consistent spin definition along a verified tree",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.assembly.export_native_assembly_figures",
      "path": "core/assembly.py",
      "line": 306,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 8 supported plotted entries.

![Original plot](assembly__spin_history.png)

Arrays: `assembly__spin_history.npz`; figure contract: `assembly__spin_history.json`.

- Actual scale factor a versus Native dimensionless spin [dimensionless]


### assembly / concentration_history

Use the same identity graph across saved epochs. Cadence limits derivative, infall and merger timing.

Required contract: `Sealed halo-main-branch-v1 stage plus exact current identity graph; actual epoch and source binding, exact IDs, original quality and cadence masks`

```json
{
  "family": "assembly",
  "variant": "concentration_history",
  "producer_route": [
    {
      "module": "core.assembly",
      "function": "export_native_assembly_figures"
    }
  ],
  "input_contract": "Sealed halo-main-branch-v1 stage plus exact current identity graph; actual epoch and source binding, exact IDs, original quality and cadence masks",
  "required_measurement": "Supported profile fits along the same tree",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.assembly.export_native_assembly_figures",
      "path": "core/assembly.py",
      "line": 306,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 8 supported plotted entries.

![Original plot](assembly__concentration_history.png)

Arrays: `assembly__concentration_history.npz`; figure contract: `assembly__concentration_history.json`.

- Actual scale factor a versus Native concentration [dimensionless]


### assembly / shape_history

Use the same identity graph across saved epochs. Cadence limits derivative, infall and merger timing.

Required contract: `Sealed halo-main-branch-v1 stage plus exact current identity graph; actual epoch and source binding, exact IDs, original quality and cadence masks`

```json
{
  "family": "assembly",
  "variant": "shape_history",
  "producer_route": [
    {
      "module": "core.assembly",
      "function": "export_native_assembly_figures"
    }
  ],
  "input_contract": "Sealed halo-main-branch-v1 stage plus exact current identity graph; actual epoch and source binding, exact IDs, original quality and cadence masks",
  "required_measurement": "Shape tensors and orientation continuity along tree",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.assembly.export_native_assembly_figures",
      "path": "core/assembly.py",
      "line": 306,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 32 supported plotted entries.

![Original plot](assembly__shape_history.png)

Arrays: `assembly__shape_history.npz`; figure contract: `assembly__shape_history.json`.

- Actual scale factor a versus Native density/potential axis ratios [dimensionless]


### assembly / pericenter_infall_merger_events

Use the same identity graph across saved epochs. Cadence limits derivative, infall and merger timing.

Required contract: `Current exact native graph with original snapshot/catalogue sources and measured quality; bound actual clocks; hash-bound recorded images and original host radius definition; explicit cadence/support flags`

```json
{
  "family": "assembly",
  "variant": "pericenter_infall_merger_events",
  "producer_route": [
    {
      "module": "core.flagship_orbits",
      "function": "measure_native_assembly_orbits"
    },
    {
      "module": "core.flagship_orbits",
      "function": "export_native_assembly_event_figures"
    }
  ],
  "input_contract": "Current exact native graph with original snapshot/catalogue sources and measured quality; bound actual clocks; hash-bound recorded images and original host radius definition; explicit cadence/support flags",
  "required_measurement": "Worldline extrema, infall definitions and actual merger links",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_orbits.measure_native_assembly_orbits",
      "path": "core/flagship_orbits.py",
      "line": 96,
      "role": "producer"
    },
    {
      "name": "core.flagship_orbits.export_native_assembly_event_figures",
      "path": "core/flagship_orbits.py",
      "line": 162,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 2 supported plotted entries.

![Original plot](assembly__pericenter_infall_merger_events.png)

Arrays: `assembly__pericenter_infall_merger_events.npz`; figure contract: `assembly__pericenter_infall_merger_events.json`.

- Actual proper time [Myr] versus Event kind [dimensionless]


### assembly / stellar_assembly

Use the same identity graph across saved epochs. Cadence limits derivative, infall and merger timing.

Required contract: `Sealed halo-main-branch-v1 stage plus exact current identity graph; actual epoch and source binding, exact IDs, original quality and cadence masks`

```json
{
  "family": "assembly",
  "variant": "stellar_assembly",
  "producer_route": [
    {
      "module": "core.assembly",
      "function": "export_native_assembly_figures"
    }
  ],
  "input_contract": "Sealed halo-main-branch-v1 stage plus exact current identity graph; actual epoch and source binding, exact IDs, original quality and cadence masks",
  "required_measurement": "Star birth IDs, in-situ/ex-situ membership and birth times",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.assembly.export_native_assembly_figures",
      "path": "core/assembly.py",
      "line": 306,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 8 supported plotted entries.

![Original plot](assembly__stellar_assembly.png)

Arrays: `assembly__stellar_assembly.npz`; figure contract: `assembly__stellar_assembly.json`.

- Actual scale factor a versus Native stellar mass [Msun]


### assembly / quenching_morphology

Use the same identity graph across saved epochs. Cadence limits derivative, infall and merger timing.

Required contract: `Sealed halo-main-branch-v1 stage plus exact current identity graph; actual epoch and source binding, exact IDs, original quality and cadence masks`

```json
{
  "family": "assembly",
  "variant": "quenching_morphology",
  "producer_route": [
    {
      "module": "core.assembly",
      "function": "export_native_assembly_figures"
    }
  ],
  "input_contract": "Sealed halo-main-branch-v1 stage plus exact current identity graph; actual epoch and source binding, exact IDs, original quality and cadence masks",
  "required_measurement": "SFR history, morphology estimator and frozen quenching definition",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.assembly.export_native_assembly_figures",
      "path": "core/assembly.py",
      "line": 306,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 24 supported plotted entries.

![Original plot](assembly__quenching_morphology.png)

Arrays: `assembly__quenching_morphology.npz`; figure contract: `assembly__quenching_morphology.json`.

- Actual scale factor a versus Native specific star formation rate [1 / year]
- Actual scale factor a versus Native stellar tensor axis ratios [dimensionless]


### web / filament_map

Read the original numeric contract, units and masks.

Required contract: `Verified native-environment-v1 original periodic T-web nodes/labels and native halo structure joins; optional sealed native graph arcs/recorded images and unique-assignment masks remain distinct from T-web`

```json
{
  "family": "web",
  "variant": "filament_map",
  "producer_route": [
    {
      "module": "core.halo_products",
      "function": "export_native_web_figures"
    }
  ],
  "input_contract": "Verified native-environment-v1 original periodic T-web nodes/labels and native halo structure joins; optional sealed native graph arcs/recorded images and unique-assignment masks remain distinct from T-web",
  "required_measurement": "Periodic density field, smoothing scale and web classifier",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.halo_products.export_native_web_figures",
      "path": "core/halo_products.py",
      "line": 390,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 18128 supported plotted entries.

![Original plot](web__filament_map.png)

Arrays: `web__filament_map.npz`; figure contract: `web__filament_map.json`.

- x [comoving kpc] versus y [comoving kpc]


### web / spin_alignment

Read the original numeric contract, units and masks.

Required contract: `Verified native-environment-v1 original periodic T-web nodes/labels and native halo structure joins; optional sealed native graph arcs/recorded images and unique-assignment masks remain distinct from T-web`

```json
{
  "family": "web",
  "variant": "spin_alignment",
  "producer_route": [
    {
      "module": "core.halo_products",
      "function": "export_native_web_figures"
    }
  ],
  "input_contract": "Verified native-environment-v1 original periodic T-web nodes/labels and native halo structure joins; optional sealed native graph arcs/recorded images and unique-assignment masks remain distinct from T-web",
  "required_measurement": "Native spins and local filament axes with selection masks",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.halo_products.export_native_web_figures",
      "path": "core/halo_products.py",
      "line": 390,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 8 supported plotted entries.

![Original plot](web__spin_alignment.png)

Arrays: `web__spin_alignment.npz`; figure contract: `web__spin_alignment.json`.

- Outer aperture radius [proper kpc] versus Unsigned cosine [dimensionless]


### web / shape_alignment

Read the original numeric contract, units and masks.

Required contract: `Verified native-environment-v1 original periodic T-web nodes/labels and native halo structure joins; optional sealed native graph arcs/recorded images and unique-assignment masks remain distinct from T-web`

```json
{
  "family": "web",
  "variant": "shape_alignment",
  "producer_route": [
    {
      "module": "core.halo_products",
      "function": "export_native_web_figures"
    }
  ],
  "input_contract": "Verified native-environment-v1 original periodic T-web nodes/labels and native halo structure joins; optional sealed native graph arcs/recorded images and unique-assignment masks remain distinct from T-web",
  "required_measurement": "Native shape axes and web eigenvectors",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.halo_products.export_native_web_figures",
      "path": "core/halo_products.py",
      "line": 390,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 8 supported plotted entries.

![Original plot](web__shape_alignment.png)

Arrays: `web__shape_alignment.npz`; figure contract: `web__shape_alignment.json`.

- Outer aperture radius [proper kpc] versus Unsigned cosine [dimensionless]


### web / web_evolution

Read the original numeric contract, units and masks.

Required contract: `Verified native-environment-history-v1 with original persistent host graph nodes, saved proper-time brackets, unoriented axis changes and clock-bounded accepted intervals`

```json
{
  "family": "web",
  "variant": "web_evolution",
  "producer_route": [
    {
      "module": "core.halo_products",
      "function": "export_native_web_history"
    }
  ],
  "input_contract": "Verified native-environment-history-v1 with original persistent host graph nodes, saved proper-time brackets, unoriented axis changes and clock-bounded accepted intervals",
  "required_measurement": "Same web definition and smoothing across saved times",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.halo_products.export_native_web_history",
      "path": "core/halo_products.py",
      "line": 493,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 1 supported plotted entries.

![Original plot](web__web_evolution.png)

Arrays: `web__web_evolution.npz`; figure contract: `web__web_evolution.json`.

- Saved interval end [Myr] versus Unsigned axis change [degrees]


### streams / stream_progenitor_3d

Check proper positions, peculiar velocities, tracer weights, tail/wrap selection, track coordinates and supported bins.

Required contract: `Native stellar IDs, progenitor membership and 3D samples; exact source epoch/namespace, physical quantities, original support and uncertainty must be declared in measured numeric product`

```json
{
  "family": "streams",
  "variant": "stream_progenitor_3d",
  "producer_route": [
    {
      "module": "core.stream.measurements",
      "function": "export_stream_support_figures"
    }
  ],
  "input_contract": "Native stellar IDs, progenitor membership and 3D samples; exact source epoch/namespace, physical quantities, original support and uncertainty must be declared in measured numeric product",
  "required_measurement": "Native stellar IDs, progenitor membership and 3D samples",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.stream.measurements.export_stream_support_figures",
      "path": "core/stream/measurements.py",
      "line": 505,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 32 supported plotted entries.

![Original plot](streams__stream_progenitor_3d.png)

Arrays: `streams__stream_progenitor_3d.npz`; figure contract: `streams__stream_progenitor_3d.json`.

- x [proper kpc] versus y [proper kpc]


### streams / intrinsic_track

Check proper positions, peculiar velocities, tracer weights, tail/wrap selection, track coordinates and supported bins.

Required contract: `stream-coordinates-v1; measured wraps and per-dataset numeric_data contracts; bootstrap and support arrays`

```json
{
  "family": "streams",
  "variant": "intrinsic_track",
  "producer_route": [
    {
      "module": "core.flagship_products",
      "function": "export_registered_stream_figures"
    }
  ],
  "input_contract": "stream-coordinates-v1; measured wraps and per-dataset numeric_data contracts; bootstrap and support arrays",
  "required_measurement": "Supported intrinsic principal curve and arc length",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_products.export_registered_stream_figures",
      "path": "core/flagship_products.py",
      "line": 648,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 74 supported plotted entries.

![Original plot](streams__intrinsic_track.png)

Arrays: `streams__intrinsic_track.npz`; figure contract: `streams__intrinsic_track.json`.

- curve_position_kpc [proper kpc] versus curve_position_kpc [proper kpc]


### streams / projected_track

Check proper positions, peculiar velocities, tracer weights, tail/wrap selection, track coordinates and supported bins.

Required contract: `stream-coordinates-v1 → existing stream geometry/observer stage; explicit viewing basis, host center/covariance, local widths and support`

```json
{
  "family": "streams",
  "variant": "projected_track",
  "producer_route": [
    {
      "module": "core.flagship_products",
      "function": "derive_native_stream_product"
    },
    {
      "module": "core.flagship_products",
      "function": "export_registered_stream_projection_figures"
    }
  ],
  "input_contract": "stream-coordinates-v1 \u2192 existing stream geometry/observer stage; explicit viewing basis, host center/covariance, local widths and support",
  "required_measurement": "Declared observer, units and projection of native track",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_products.derive_native_stream_product",
      "path": "core/flagship_products.py",
      "line": 619,
      "role": "producer"
    },
    {
      "name": "core.flagship_products.export_registered_stream_projection_figures",
      "path": "core/flagship_products.py",
      "line": 730,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 26 supported plotted entries.

![Original plot](streams__projected_track.png)

Arrays: `streams__projected_track.npz`; figure contract: `streams__projected_track.json`.

- view_000_projected_kpc [proper kpc] versus view_000_projected_kpc [proper kpc]


### streams / curvature

Check proper positions, peculiar velocities, tracer weights, tail/wrap selection, track coordinates and supported bins.

Required contract: `stream-coordinates-v1 → existing stream geometry/observer stage; explicit viewing basis, host center/covariance, local widths and support`

```json
{
  "family": "streams",
  "variant": "curvature",
  "producer_route": [
    {
      "module": "core.flagship_products",
      "function": "derive_native_stream_product"
    },
    {
      "module": "core.flagship_products",
      "function": "export_registered_stream_projection_figures"
    }
  ],
  "input_contract": "stream-coordinates-v1 \u2192 existing stream geometry/observer stage; explicit viewing basis, host center/covariance, local widths and support",
  "required_measurement": "Track derivatives with curvature uncertainty",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_products.derive_native_stream_product",
      "path": "core/flagship_products.py",
      "line": 619,
      "role": "producer"
    },
    {
      "name": "core.flagship_products.export_registered_stream_projection_figures",
      "path": "core/flagship_products.py",
      "line": 730,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 36 supported plotted entries.

![Original plot](streams__curvature.png)

Arrays: `streams__curvature.npz`; figure contract: `streams__curvature.json`.

- view_000_projected_arclength_kpc [proper kpc] versus view_000_curvature_kpc_inverse [1 / proper kpc]


### streams / convexity

Check proper positions, peculiar velocities, tracer weights, tail/wrap selection, track coordinates and supported bins.

Required contract: `stream-coordinates-v1 → existing stream geometry/observer stage; explicit viewing basis, host center/covariance, local widths and support`

```json
{
  "family": "streams",
  "variant": "convexity",
  "producer_route": [
    {
      "module": "core.flagship_products",
      "function": "derive_native_stream_product"
    },
    {
      "module": "core.flagship_products",
      "function": "export_registered_stream_projection_figures"
    }
  ],
  "input_contract": "stream-coordinates-v1 \u2192 existing stream geometry/observer stage; explicit viewing basis, host center/covariance, local widths and support",
  "required_measurement": "Track orientation and signed-convexity convention",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_products.derive_native_stream_product",
      "path": "core/flagship_products.py",
      "line": 619,
      "role": "producer"
    },
    {
      "name": "core.flagship_products.export_registered_stream_projection_figures",
      "path": "core/flagship_products.py",
      "line": 730,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 26 supported plotted entries.

![Original plot](streams__convexity.png)

Arrays: `streams__convexity.npz`; figure contract: `streams__convexity.json`.

- view_000_projected_arclength_kpc [proper kpc] versus view_000_convex_probability [dimensionless]


### streams / linear_density

Check proper positions, peculiar velocities, tracer weights, tail/wrap selection, track coordinates and supported bins.

Required contract: `stream-coordinates-v1; measured wraps and per-dataset numeric_data contracts; bootstrap and support arrays`

```json
{
  "family": "streams",
  "variant": "linear_density",
  "producer_route": [
    {
      "module": "core.flagship_products",
      "function": "export_registered_stream_figures"
    }
  ],
  "input_contract": "stream-coordinates-v1; measured wraps and per-dataset numeric_data contracts; bootstrap and support arrays",
  "required_measurement": "Selection-corrected counts/mass per arc-length bin",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_products.export_registered_stream_figures",
      "path": "core/flagship_products.py",
      "line": 648,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 9 supported plotted entries.

![Original plot](streams__linear_density.png)

Arrays: `streams__linear_density.npz`; figure contract: `streams__linear_density.json`.

- bin_center_kpc [proper kpc] versus linear_density_msun_kpc [Msun / proper kpc]


### streams / width

Check proper positions, peculiar velocities, tracer weights, tail/wrap selection, track coordinates and supported bins.

Required contract: `stream-coordinates-v1; measured wraps and per-dataset numeric_data contracts; bootstrap and support arrays`

```json
{
  "family": "streams",
  "variant": "width",
  "producer_route": [
    {
      "module": "core.flagship_products",
      "function": "export_registered_stream_figures"
    }
  ],
  "input_contract": "stream-coordinates-v1; measured wraps and per-dataset numeric_data contracts; bootstrap and support arrays",
  "required_measurement": "Transverse distribution, estimator and unresolved-width masks",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_products.export_registered_stream_figures",
      "path": "core/flagship_products.py",
      "line": 648,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 18 supported plotted entries.

![Original plot](streams__width.png)

Arrays: `streams__width.npz`; figure contract: `streams__width.json`.

- bin_center_kpc [proper kpc] versus width_axes_kpc [proper kpc]


### streams / velocity

Check proper positions, peculiar velocities, tracer weights, tail/wrap selection, track coordinates and supported bins.

Required contract: `stream-coordinates-v1; measured wraps and per-dataset numeric_data contracts; bootstrap and support arrays`

```json
{
  "family": "streams",
  "variant": "velocity",
  "producer_route": [
    {
      "module": "core.flagship_products",
      "function": "export_registered_stream_figures"
    }
  ],
  "input_contract": "stream-coordinates-v1; measured wraps and per-dataset numeric_data contracts; bootstrap and support arrays",
  "required_measurement": "Stream-frame velocity mean and component definitions",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_products.export_registered_stream_figures",
      "path": "core/flagship_products.py",
      "line": 648,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 27 supported plotted entries.

![Original plot](streams__velocity.png)

Arrays: `streams__velocity.npz`; figure contract: `streams__velocity.json`.

- bin_center_kpc [proper kpc] versus velocity_mean_kms [km/s]


### streams / dispersion

Check proper positions, peculiar velocities, tracer weights, tail/wrap selection, track coordinates and supported bins.

Required contract: `stream-coordinates-v1; measured wraps and per-dataset numeric_data contracts; bootstrap and support arrays`

```json
{
  "family": "streams",
  "variant": "dispersion",
  "producer_route": [
    {
      "module": "core.flagship_products",
      "function": "export_registered_stream_figures"
    }
  ],
  "input_contract": "stream-coordinates-v1; measured wraps and per-dataset numeric_data contracts; bootstrap and support arrays",
  "required_measurement": "Intrinsic velocity scatter, sampling uncertainty and masks",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_products.export_registered_stream_figures",
      "path": "core/flagship_products.py",
      "line": 648,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 27 supported plotted entries.

![Original plot](streams__dispersion.png)

Arrays: `streams__dispersion.npz`; figure contract: `streams__dispersion.json`.

- bin_center_kpc [proper kpc] versus velocity_dispersion_kms [km / s]


### streams / gaps

Check proper positions, peculiar velocities, tracer weights, tail/wrap selection, track coordinates and supported bins.

Required contract: `stream-coordinates-v1; measured wraps and per-dataset numeric_data contracts; bootstrap and support arrays`

```json
{
  "family": "streams",
  "variant": "gaps",
  "producer_route": [
    {
      "module": "core.flagship_products",
      "function": "export_registered_stream_figures"
    }
  ],
  "input_contract": "stream-coordinates-v1; measured wraps and per-dataset numeric_data contracts; bootstrap and support arrays",
  "required_measurement": "Matched reference density, gap definition and significance",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_products.export_registered_stream_figures",
      "path": "core/flagship_products.py",
      "line": 648,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 9 supported plotted entries.

![Original plot](streams__gaps.png)

Arrays: `streams__gaps.npz`; figure contract: `streams__gaps.json`.

- bin_center_kpc [proper kpc] versus density_feature_significance [dimensionless]


### streams / spurs

Check proper positions, peculiar velocities, tracer weights, tail/wrap selection, track coordinates and supported bins.

Required contract: `stream-coordinates-v1; measured wraps and per-dataset numeric_data contracts; bootstrap and support arrays`

```json
{
  "family": "streams",
  "variant": "spurs",
  "producer_route": [
    {
      "module": "core.flagship_products",
      "function": "export_registered_stream_figures"
    }
  ],
  "input_contract": "stream-coordinates-v1; measured wraps and per-dataset numeric_data contracts; bootstrap and support arrays",
  "required_measurement": "Supported transverse residual peaks against matched track",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_products.export_registered_stream_figures",
      "path": "core/flagship_products.py",
      "line": 648,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 18 supported plotted entries.

![Original plot](streams__spurs.png)

Arrays: `streams__spurs.npz`; figure contract: `streams__spurs.json`.

- bin_center_kpc [proper kpc] versus transverse_mean_kpc [proper kpc]


### streams / bifurcations

Check proper positions, peculiar velocities, tracer weights, tail/wrap selection, track coordinates and supported bins.

Required contract: `Original local two-mode fork proposals; persistent physical branches and causal encounters not established`

```json
{
  "family": "streams",
  "variant": "bifurcations",
  "producer_route": [
    {
      "module": "core.stream.measurements",
      "function": "measured_track_support_products"
    },
    {
      "module": "core.flagship_products",
      "function": "export_registered_stream_figures"
    }
  ],
  "input_contract": "Original local two-mode fork proposals; persistent physical branches and causal encounters not established",
  "required_measurement": "Separate wrap/branch membership and persistence evidence",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.stream.measurements.measured_track_support_products",
      "path": "core/stream/measurements.py",
      "line": 430,
      "role": "producer"
    },
    {
      "name": "core.flagship_products.export_registered_stream_figures",
      "path": "core/flagship_products.py",
      "line": 648,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 2 supported plotted entries.

![Original plot](streams__bifurcations.png)

Arrays: `streams__bifurcations.npz`; figure contract: `streams__bifurcations.json`.

- bin_center_kpc [proper kpc] versus bifurcation_separation_kpc [proper kpc]


### streams / action_angle

Check proper positions, peculiar velocities, tracer weights, tail/wrap selection, track coordinates and supported bins.

Required contract: `Supported potential domain, action solver and error controls; exact source epoch/namespace, physical quantities, original support and uncertainty must be declared in measured numeric product`

```json
{
  "family": "streams",
  "variant": "action_angle",
  "producer_route": [
    {
      "module": "core.stream.measurements",
      "function": "export_stream_support_figures"
    }
  ],
  "input_contract": "Supported potential domain, action solver and error controls; exact source epoch/namespace, physical quantities, original support and uncertainty must be declared in measured numeric product",
  "required_measurement": "Supported potential domain, action solver and error controls",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.stream.measurements.export_stream_support_figures",
      "path": "core/stream/measurements.py",
      "line": 505,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 27 supported plotted entries.

![Original plot](streams__action_angle.png)

Arrays: `streams__action_angle.npz`; figure contract: `streams__action_angle.json`.

- angle [radian] versus action [proper kpc km/s]


### streams / density_power

Check proper positions, peculiar velocities, tracer weights, tail/wrap selection, track coordinates and supported bins.

Required contract: `stream-coordinates-v1; measured wraps and per-dataset numeric_data contracts; bootstrap and support arrays`

```json
{
  "family": "streams",
  "variant": "density_power",
  "producer_route": [
    {
      "module": "core.flagship_products",
      "function": "export_registered_stream_figures"
    }
  ],
  "input_contract": "stream-coordinates-v1; measured wraps and per-dataset numeric_data contracts; bootstrap and support arrays",
  "required_measurement": "Density spectrum with window/noise treatment",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_products.export_registered_stream_figures",
      "path": "core/flagship_products.py",
      "line": 648,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 9 supported plotted entries.

![Original plot](streams__density_power.png)

Arrays: `streams__density_power.npz`; figure contract: `streams__density_power.json`.

- wavenumber_kpc_inverse [1 / proper kpc] versus density_power_kpc [proper kpc]


### streams / track_power

Check proper positions, peculiar velocities, tracer weights, tail/wrap selection, track coordinates and supported bins.

Required contract: `Raw two-component transverse bin-centroid residual power around fitted curve, Hann/support window, separately published conditional noise from width covariance/effective count; properkpc^3. No noise/baseline fit or subtraction.`

```json
{
  "family": "streams",
  "variant": "track_power",
  "producer_route": [
    {
      "module": "core.stream.measurements",
      "function": "measured_track_support_products"
    },
    {
      "module": "core.flagship_products",
      "function": "export_registered_stream_figures"
    }
  ],
  "input_contract": "Raw two-component transverse bin-centroid residual power around fitted curve, Hann/support window, separately published conditional noise from width covariance/effective count; properkpc^3. No noise/baseline fit or subtraction.",
  "required_measurement": "Track residual spectrum with coordinate and window controls",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.stream.measurements.measured_track_support_products",
      "path": "core/stream/measurements.py",
      "line": 430,
      "role": "producer"
    },
    {
      "name": "core.flagship_products.export_registered_stream_figures",
      "path": "core/flagship_products.py",
      "line": 648,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 18 supported plotted entries.

![Original plot](streams__track_power.png)

Arrays: `streams__track_power.npz`; figure contract: `streams__track_power.json`.

- wavenumber_kpc_inverse [1 / proper kpc] versus track_power_kpc3 [proper kpc^3]


### encounters / worldlines_3d

Use common frames, actual trajectories, interpolation certificates and measured impact geometry. Close passage is not proof of a causal gap.

Required contract: `native-encounter-frame-history-v1 with explicit interval flags and frame-definition hash; stream-encounters-v1 preserves refinement requests and unresolved intervals`

```json
{
  "family": "encounters",
  "variant": "worldlines_3d",
  "producer_route": [
    {
      "module": "core.flagship_products",
      "function": "native_encounter_frame"
    },
    {
      "module": "core.flagship_products",
      "function": "measure_native_encounters"
    },
    {
      "module": "core.flagship_products",
      "function": "export_native_encounter_figures"
    }
  ],
  "input_contract": "native-encounter-frame-history-v1 with explicit interval flags and frame-definition hash; stream-encounters-v1 preserves refinement requests and unresolved intervals",
  "required_measurement": "Persistent perturber IDs and stream worldlines in one frame",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_products.native_encounter_frame",
      "path": "core/flagship_products.py",
      "line": 413,
      "role": "producer"
    },
    {
      "name": "core.flagship_products.measure_native_encounters",
      "path": "core/flagship_products.py",
      "line": 503,
      "role": "producer"
    },
    {
      "name": "core.flagship_products.export_native_encounter_figures",
      "path": "core/flagship_products.py",
      "line": 541,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 9 supported plotted entries.

![Original plot](encounters__worldlines_3d.png)

Arrays: `encounters__worldlines_3d.npz`; figure contract: `encounters__worldlines_3d.json`.

- x [proper kpc] versus y [proper kpc]


### encounters / impact_velocity_geometry

Use common frames, actual trajectories, interpolation certificates and measured impact geometry. Close passage is not proof of a causal gap.

Required contract: `native-encounter-frame-history-v1 with explicit interval flags and frame-definition hash; stream-encounters-v1 preserves refinement requests and unresolved intervals`

```json
{
  "family": "encounters",
  "variant": "impact_velocity_geometry",
  "producer_route": [
    {
      "module": "core.flagship_products",
      "function": "native_encounter_frame"
    },
    {
      "module": "core.flagship_products",
      "function": "measure_native_encounters"
    },
    {
      "module": "core.flagship_products",
      "function": "export_native_encounter_figures"
    }
  ],
  "input_contract": "native-encounter-frame-history-v1 with explicit interval flags and frame-definition hash; stream-encounters-v1 preserves refinement requests and unresolved intervals",
  "required_measurement": "Certified closest approach, relative speed and orientation",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_products.native_encounter_frame",
      "path": "core/flagship_products.py",
      "line": 413,
      "role": "producer"
    },
    {
      "name": "core.flagship_products.measure_native_encounters",
      "path": "core/flagship_products.py",
      "line": 503,
      "role": "producer"
    },
    {
      "name": "core.flagship_products.export_native_encounter_figures",
      "path": "core/flagship_products.py",
      "line": 541,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 2 supported plotted entries.

![Original plot](encounters__impact_velocity_geometry.png)

Arrays: `encounters__impact_velocity_geometry.npz`; figure contract: `encounters__impact_velocity_geometry.json`.

- time [Myr] versus impact [proper kpc]
- speed [km / s] versus impact [proper kpc]


### encounters / perturber_mass_profile

Use common frames, actual trajectories, interpolation certificates and measured impact geometry. Close passage is not proof of a causal gap.

Required contract: `Exact native descendant graph, bound memberships, actual gravitating census, recorded images and converted stream frame`

```json
{
  "family": "encounters",
  "variant": "perturber_mass_profile",
  "producer_route": [
    {
      "module": "core.stream.perturbers",
      "function": "native_perturber_history_runner"
    }
  ],
  "input_contract": "Exact native descendant graph, bound memberships, actual gravitating census, recorded images and converted stream frame",
  "required_measurement": "Time-dependent bound profile and mass-definition support",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.stream.perturbers.native_perturber_history_runner",
      "path": "core/stream/perturbers.py",
      "line": 64,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 21 supported plotted entries.

![Original plot](encounters__perturber_mass_profile.png)

Arrays: `encounters__perturber_mass_profile.npz`; figure contract: `encounters__perturber_mass_profile.json`.

- Radius [proper kpc] versus Enclosed bound mass [Msun]


### encounters / before_after_controls

Use common frames, actual trajectories, interpolation certificates and measured impact geometry. Close passage is not proof of a causal gap.

Required contract: `Original native paired comparison replay, exact selected stellar cohort, same source-bound physical chart and optional measured host actions`

```json
{
  "family": "encounters",
  "variant": "before_after_controls",
  "producer_route": [
    {
      "module": "core.native_stream_responses",
      "function": "native_response_runner"
    }
  ],
  "input_contract": "Original native paired comparison replay, exact selected stellar cohort, same source-bound physical chart and optional measured host actions",
  "required_measurement": "Same tracers, times and host with declared intervention",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.native_stream_responses.native_response_runner",
      "path": "core/native_stream_responses.py",
      "line": 134,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 56 supported plotted entries.

![Original plot](encounters__before_after_controls.png)

Arrays: `encounters__before_after_controls.npz`; figure contract: `encounters__before_after_controls.json`.

- Frozen reference arclength [proper kpc] versus Matched line densities [Msun / proper kpc]
- Frozen reference arclength [proper kpc] versus Matched transverse offsets [proper kpc]
- Frozen reference arclength [proper kpc] versus Matched local velocity changes [km/s]


### encounters / gap_growth

Use common frames, actual trajectories, interpolation certificates and measured impact geometry. Close passage is not proof of a causal gap.

Required contract: `Original native paired comparison replay, exact selected stellar cohort, same source-bound physical chart and optional measured host actions`

```json
{
  "family": "encounters",
  "variant": "gap_growth",
  "producer_route": [
    {
      "module": "core.native_stream_responses",
      "function": "native_response_runner"
    }
  ],
  "input_contract": "Original native paired comparison replay, exact selected stellar cohort, same source-bound physical chart and optional measured host actions",
  "required_measurement": "Same gap definition through time and matched baseline",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.native_stream_responses.native_response_runner",
      "path": "core/native_stream_responses.py",
      "line": 134,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 12 supported plotted entries.

![Original plot](encounters__gap_growth.png)

Arrays: `encounters__gap_growth.npz`; figure contract: `encounters__gap_growth.json`.

- Saved time [Myr] versus gap_growth [dimensionless]


### encounters / action_diffusion

Use common frames, actual trajectories, interpolation certificates and measured impact geometry. Close passage is not proof of a causal gap.

Required contract: `Original native paired comparison replay, exact selected stellar cohort, same source-bound physical chart and optional measured host actions`

```json
{
  "family": "encounters",
  "variant": "action_diffusion",
  "producer_route": [
    {
      "module": "core.native_stream_responses",
      "function": "native_response_runner"
    }
  ],
  "input_contract": "Original native paired comparison replay, exact selected stellar cohort, same source-bound physical chart and optional measured host actions",
  "required_measurement": "Supported action histories and matched stochastic/control variance",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.native_stream_responses.native_response_runner",
      "path": "core/native_stream_responses.py",
      "line": 134,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 9 supported plotted entries.

![Original plot](encounters__action_diffusion.png)

Arrays: `encounters__action_diffusion.npz`; figure contract: `encounters__action_diffusion.json`.

- Saved time [Myr] versus Mean squared paired action change [proper kpc^2
(km/s)^2]


### encounters / bh_extended_subhalo_comparison

Use common frames, actual trajectories, interpolation certificates and measured impact geometry. Close passage is not proof of a causal gap.

Required contract: `Original executed matched imposed-field templates with exact common design/realization/initial cohort/geometry/bins`

```json
{
  "family": "encounters",
  "variant": "bh_extended_subhalo_comparison",
  "producer_route": [
    {
      "module": "core.matched_perturber_figures",
      "function": "matched_template_figures_runner"
    }
  ],
  "input_contract": "Original executed matched imposed-field templates with exact common design/realization/initial cohort/geometry/bins",
  "required_measurement": "Matched compact/extended perturber masses, orbits and force resolution",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.matched_perturber_figures.matched_template_figures_runner",
      "path": "core/matched_perturber_figures.py",
      "line": 35,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 32 supported plotted entries.

![Original plot](encounters__bh_extended_subhalo_comparison.png)

Arrays: `encounters__bh_extended_subhalo_comparison.npz`; figure contract: `encounters__bh_extended_subhalo_comparison.json`.

- Frozen track arclength [proper kpc] versus Matched final line density [Msun / proper kpc]
- Frozen track arclength [proper kpc] versus Final transverse centroid difference [proper kpc]
- Frozen track arclength [proper kpc] versus Final local velocity difference [km/s]


### black_holes / bh_growth

Separate gravity mass, subgrid BH mass, accretion, feedback and committed source events. A marker is not an event horizon.

Required contract: `Current source-bound native BH history with exact namespace/ID/epoch and original field-summary equality; merger timeline requires independently replayed native-hydro-history and complete committed BH journal`

```json
{
  "family": "black_holes",
  "variant": "bh_growth",
  "producer_route": [
    {
      "module": "core.flagship_bh",
      "function": "export_native_bh_figures"
    }
  ],
  "input_contract": "Current source-bound native BH history with exact namespace/ID/epoch and original field-summary equality; merger timeline requires independently replayed native-hydro-history and complete committed BH journal",
  "required_measurement": "Persistent native BH IDs, mass and saved times",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_bh.export_native_bh_figures",
      "path": "core/flagship_bh.py",
      "line": 38,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 6 supported plotted entries.

![Original plot](black_holes__bh_growth.png)

Arrays: `black_holes__bh_growth.npz`; figure contract: `black_holes__bh_growth.json`.

- Actual scale factor a versus Subgrid BH mass [Msun]


### black_holes / eddington_ratio

Separate gravity mass, subgrid BH mass, accretion, feedback and committed source events. A marker is not an event horizon.

Required contract: `Current source-bound native BH history with exact namespace/ID/epoch and original field-summary equality; merger timeline requires independently replayed native-hydro-history and complete committed BH journal`

```json
{
  "family": "black_holes",
  "variant": "eddington_ratio",
  "producer_route": [
    {
      "module": "core.flagship_bh",
      "function": "export_native_bh_figures"
    }
  ],
  "input_contract": "Current source-bound native BH history with exact namespace/ID/epoch and original field-summary equality; merger timeline requires independently replayed native-hydro-history and complete committed BH journal",
  "required_measurement": "Native accretion rate and consistent Eddington convention",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_bh.export_native_bh_figures",
      "path": "core/flagship_bh.py",
      "line": 38,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 6 supported plotted entries.

![Original plot](black_holes__eddington_ratio.png)

Arrays: `black_holes__eddington_ratio.npz`; figure contract: `black_holes__eddington_ratio.json`.

- Actual scale factor a versus Mean inflow / saved-mass Eddington cap [dimensionless]


### black_holes / pair_separation

Separate gravity mass, subgrid BH mass, accretion, feedback and committed source events. A marker is not an event horizon.

Required contract: `Current native BH history; exact two IDs/namespace; original saved proper coordinates/epochs and periodic build; missingness retained`

```json
{
  "family": "black_holes",
  "variant": "pair_separation",
  "producer_route": [
    {
      "module": "core.flagship_bh",
      "function": "export_native_bh_pair_separation"
    }
  ],
  "input_contract": "Current native BH history; exact two IDs/namespace; original saved proper coordinates/epochs and periodic build; missingness retained",
  "required_measurement": "Actual BH pair identities and proper separation",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_bh.export_native_bh_pair_separation",
      "path": "core/flagship_bh.py",
      "line": 184,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 6 supported plotted entries.

![Original plot](black_holes__pair_separation.png)

Arrays: `black_holes__pair_separation.npz`; figure contract: `black_holes__pair_separation.json`.

- Actual scale factor a versus Separation [proper kpc]


### black_holes / merger_timeline

Separate gravity mass, subgrid BH mass, accretion, feedback and committed source events. A marker is not an event horizon.

Required contract: `Current source-bound native BH history with exact namespace/ID/epoch and original field-summary equality; merger timeline requires independently replayed native-hydro-history and complete committed BH journal`

```json
{
  "family": "black_holes",
  "variant": "merger_timeline",
  "producer_route": [
    {
      "module": "core.flagship_bh",
      "function": "export_native_bh_merger_timeline"
    }
  ],
  "input_contract": "Current source-bound native BH history with exact namespace/ID/epoch and original field-summary equality; merger timeline requires independently replayed native-hydro-history and complete committed BH journal",
  "required_measurement": "Actual merger event records and ancestry",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_bh.export_native_bh_merger_timeline",
      "path": "core/flagship_bh.py",
      "line": 146,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 1 supported plotted entries.

![Original plot](black_holes__merger_timeline.png)

Arrays: `black_holes__merger_timeline.npz`; figure contract: `black_holes__merger_timeline.json`.

- Applied proper time [Myr] versus Event ordinal [dimensionless]


### black_holes / recoil_trajectory

Separate gravity mass, subgrid BH mass, accretion, feedback and committed source events. A marker is not an event horizon.

Required contract: `Current source-bound BH history, independently replayed complete native hydro journal with exact selected merger counter/survivor, hash-bound recorded images for original saved epochs, actual clock and later ancestry masks`

```json
{
  "family": "black_holes",
  "variant": "recoil_trajectory",
  "producer_route": [
    {
      "module": "core.flagship_bh",
      "function": "export_native_bh_recoil_trajectory"
    }
  ],
  "input_contract": "Current source-bound BH history, independently replayed complete native hydro journal with exact selected merger counter/survivor, hash-bound recorded images for original saved epochs, actual clock and later ancestry masks",
  "required_measurement": "Implemented kick physics and native post-kick trajectory",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_bh.export_native_bh_recoil_trajectory",
      "path": "core/flagship_bh.py",
      "line": 277,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 2 supported plotted entries.

![Original plot](black_holes__recoil_trajectory.png)

Arrays: `black_holes__recoil_trajectory.npz`; figure contract: `black_holes__recoil_trajectory.json`.

- x [proper kpc] versus y [proper kpc]


### black_holes / feedback_energy

Separate gravity mass, subgrid BH mass, accretion, feedback and committed source events. A marker is not an event horizon.

Required contract: `Current source-bound native BH history with exact namespace/ID/epoch and original field-summary equality; merger timeline requires independently replayed native-hydro-history and complete committed BH journal`

```json
{
  "family": "black_holes",
  "variant": "feedback_energy",
  "producer_route": [
    {
      "module": "core.flagship_bh",
      "function": "export_native_bh_figures"
    }
  ],
  "input_contract": "Current source-bound native BH history with exact namespace/ID/epoch and original field-summary equality; merger timeline requires independently replayed native-hydro-history and complete committed BH journal",
  "required_measurement": "Native accumulated feedback energy and time convention",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_bh.export_native_bh_figures",
      "path": "core/flagship_bh.py",
      "line": 38,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 6 supported plotted entries.

![Original plot](black_holes__feedback_energy.png)

Arrays: `black_holes__feedback_energy.npz`; figure contract: `black_holes__feedback_energy.json`.

- Actual scale factor a versus Deposited feedback energy [erg]


### black_holes / dm_density_response

Separate gravity mass, subgrid BH mass, accretion, feedback and committed source events. A marker is not an event horizon.

Required contract: `Current verified native structures; replayed reciprocal unique membership counterparts; actual phase PASS; matching actual epochs/proper bins/species/estimator; BH-only config contrast; measured full joint covariance bound to exact source hashes`

```json
{
  "family": "black_holes",
  "variant": "dm_density_response",
  "producer_route": [
    {
      "module": "core.flagship_bh_response",
      "function": "export_native_bh_dm_response"
    }
  ],
  "input_contract": "Current verified native structures; replayed reciprocal unique membership counterparts; actual phase PASS; matching actual epochs/proper bins/species/estimator; BH-only config contrast; measured full joint covariance bound to exact source hashes",
  "required_measurement": "Matched BH feedback intervention and DM profiles",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_bh_response.export_native_bh_dm_response",
      "path": "core/flagship_bh_response.py",
      "line": 132,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 2 supported plotted entries.

![Original plot](black_holes__dm_density_response.png)

Arrays: `black_holes__dm_density_response.npz`; figure contract: `black_holes__dm_density_response.json`.

- Radius [proper kpc] versus B - A [Msun/(proper kpc)^3]


### black_holes / dm_shape_response

Separate gravity mass, subgrid BH mass, accretion, feedback and committed source events. A marker is not an event horizon.

Required contract: `Current verified native structures; replayed reciprocal unique membership counterparts; actual phase PASS; matching actual epochs/proper bins/species/estimator; BH-only config contrast; measured full joint covariance bound to exact source hashes`

```json
{
  "family": "black_holes",
  "variant": "dm_shape_response",
  "producer_route": [
    {
      "module": "core.flagship_bh_response",
      "function": "export_native_bh_dm_response"
    }
  ],
  "input_contract": "Current verified native structures; replayed reciprocal unique membership counterparts; actual phase PASS; matching actual epochs/proper bins/species/estimator; BH-only config contrast; measured full joint covariance bound to exact source hashes",
  "required_measurement": "Matched feedback intervention and resolved DM shape tensors",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_bh_response.export_native_bh_dm_response",
      "path": "core/flagship_bh_response.py",
      "line": 132,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 4 supported plotted entries.

![Original plot](black_holes__dm_shape_response.png)

Arrays: `black_holes__dm_shape_response.npz`; figure contract: `black_holes__dm_shape_response.json`.

- Radius [proper kpc] versus B - A [dimensionless]


### baryons / gas_phase_diagram

Distinguish thermal gas, effective EOS, seeded SSPs, actual birth events and spectral synthesis. Missing composition must remain missing.

Required contract: `Hash-bound verified native hydro history and explicit positive physical bin edges; original availability masks and effective EOS exclusion`

```json
{
  "family": "baryons",
  "variant": "gas_phase_diagram",
  "producer_route": [
    {
      "module": "core.baryon_products",
      "function": "export_native_gas_phase"
    }
  ],
  "input_contract": "Hash-bound verified native hydro history and explicit positive physical bin edges; original availability masks and effective EOS exclusion",
  "required_measurement": "Native gas density, temperature, mass and physical units",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.baryon_products.export_native_gas_phase",
      "path": "core/baryon_products.py",
      "line": 216,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 64 supported plotted entries.

![Original plot](baryons__gas_phase_diagram.png)

Arrays: `baryons__gas_phase_diagram.npz`; figure contract: `baryons__gas_phase_diagram.json`.

- Proper density [g/cm^3] versus Temperature [K]
-  versus Gas mass per bin [Msun]


### baryons / star_formation_history

Distinguish thermal gas, effective EOS, seeded SSPs, actual birth events and spectral synthesis. Missing composition must remain missing.

Required contract: `Verified native hydro history complete gas SFR census, matching original summary, actual cosmological saved clock; existing build_star_formation_history estimator; saved gaps and zero rates retained`

```json
{
  "family": "baryons",
  "variant": "star_formation_history",
  "producer_route": [
    {
      "module": "core.baryon_products",
      "function": "export_native_sfr_history"
    }
  ],
  "input_contract": "Verified native hydro history complete gas SFR census, matching original summary, actual cosmological saved clock; existing build_star_formation_history estimator; saved gaps and zero rates retained",
  "required_measurement": "Native formation times, initial stellar masses and time bins",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.baryon_products.export_native_sfr_history",
      "path": "core/baryon_products.py",
      "line": 70,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 6 supported plotted entries.

![Original plot](baryons__star_formation_history.png)

Arrays: `baryons__star_formation_history.npz`; figure contract: `baryons__star_formation_history.json`.

- Actual scale factor [dimensionless] versus Whole snapshot instantaneous SFR [Msun/yr]


### baryons / mass_return

Distinguish thermal gas, effective EOS, seeded SSPs, actual birth events and spectral synthesis. Missing composition must remain missing.

Required contract: `Verified committed native stellar return journal and complete original snapshot counters; separate signed stellar-source loss and gas-recipient gain since first selected saved epoch; prompt bookkeeping excluded`

```json
{
  "family": "baryons",
  "variant": "mass_return",
  "producer_route": [
    {
      "module": "core.baryon_products",
      "function": "export_native_mass_return"
    }
  ],
  "input_contract": "Verified committed native stellar return journal and complete original snapshot counters; separate signed stellar-source loss and gas-recipient gain since first selected saved epoch; prompt bookkeeping excluded",
  "required_measurement": "Verified delayed-return table and native deposited mass ledger",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.baryon_products.export_native_mass_return",
      "path": "core/baryon_products.py",
      "line": 135,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 12 supported plotted entries.

![Original plot](baryons__mass_return.png)

Arrays: `baryons__mass_return.npz`; figure contract: `baryons__mass_return.json`.

- Actual scale factor [dimensionless] versus Mass transferred since first saved epoch [Msun]


### baryons / metals_history

Distinguish thermal gas, effective EOS, seeded SSPs, actual birth events and spectral synthesis. Missing composition must remain missing.

Required contract: `Independently verified native-hydro-history-v1: original eleven-element gas mass fractions and completeness, SSP initial locked birth mass/metallicity, complete original source/recipient delayed-return rows; current stellar chemical mass explicitly unavailable for nonempty census`

```json
{
  "family": "baryons",
  "variant": "metals_history",
  "producer_route": [
    {
      "module": "core.baryon_products",
      "function": "export_native_metal_history"
    }
  ],
  "input_contract": "Independently verified native-hydro-history-v1: original eleven-element gas mass fractions and completeness, SSP initial locked birth mass/metallicity, complete original source/recipient delayed-return rows; current stellar chemical mass explicitly unavailable for nonempty census",
  "required_measurement": "Element-resolved stellar/gas metal ledgers and closure",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.baryon_products.export_native_metal_history",
      "path": "core/baryon_products.py",
      "line": 318,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 24 supported plotted entries.

![Original plot](baryons__metals_history.png)

Arrays: `baryons__metals_history.npz`; figure contract: `baryons__metals_history.json`.

- Actual scale factor [dimensionless] versus Metal mass [Msun]
- Actual scale factor [dimensionless] versus Metal mass [Msun]
- Actual scale factor [dimensionless] versus Metal mass [Msun]


### baryons / luminosity_history

Distinguish thermal gas, effective EOS, seeded SSPs, actual birth events and spectral synthesis. Missing composition must remain missing.

Required contract: `native-population-history-request-v1 with current hash-bound native hydro history and original pinned FSPS synthesis configuration/filter assets; all native species-4 birth states, actual proper clock and exact namespace; incomplete SSP support masks whole-census totals; rest-frame intrinsic AB colors from summed band Lnu`

```json
{
  "family": "baryons",
  "variant": "luminosity_history",
  "producer_route": [
    {
      "module": "core.baryon_products",
      "function": "export_native_population_history"
    }
  ],
  "input_contract": "native-population-history-request-v1 with current hash-bound native hydro history and original pinned FSPS synthesis configuration/filter assets; all native species-4 birth states, actual proper clock and exact namespace; incomplete SSP support masks whole-census totals; rest-frame intrinsic AB colors from summed band Lnu",
  "required_measurement": "Pinned SPS model, native age/Z/mass and filter response",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.baryon_products.export_native_population_history",
      "path": "core/baryon_products.py",
      "line": 394,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 6 supported plotted entries.

![Original plot](baryons__luminosity_history.png)

Arrays: `baryons__luminosity_history.npz`; figure contract: `baryons__luminosity_history.json`.

- Actual saved proper time [Myr] versus bolometric_total [Lsun]


### baryons / color_history

Distinguish thermal gas, effective EOS, seeded SSPs, actual birth events and spectral synthesis. Missing composition must remain missing.

Required contract: `native-population-history-request-v1 with current hash-bound native hydro history and original pinned FSPS synthesis configuration/filter assets; all native species-4 birth states, actual proper clock and exact namespace; incomplete SSP support masks whole-census totals; rest-frame intrinsic AB colors from summed band Lnu`

```json
{
  "family": "baryons",
  "variant": "color_history",
  "producer_route": [
    {
      "module": "core.baryon_products",
      "function": "export_native_population_history"
    }
  ],
  "input_contract": "native-population-history-request-v1 with current hash-bound native hydro history and original pinned FSPS synthesis configuration/filter assets; all native species-4 birth states, actual proper clock and exact namespace; incomplete SSP support masks whole-census totals; rest-frame intrinsic AB colors from summed band Lnu",
  "required_measurement": "Same SPS assumptions and explicit passbands",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.baryon_products.export_native_population_history",
      "path": "core/baryon_products.py",
      "line": 394,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 6 supported plotted entries.

![Original plot](baryons__color_history.png)

Arrays: `baryons__color_history.npz`; figure contract: `baryons__color_history.json`.

- Actual saved proper time [Myr] versus color_ab [AB mag]


### baryons / first_cooling_birth

Distinguish thermal gas, effective EOS, seeded SSPs, actual birth events and spectral synthesis. Missing composition must remain missing.

Required contract: `Current source-bound original first-star-environments-v1; original cooling/birth numeric times match rows, exact namespace identities, saved interval and host ancestry brackets retained`

```json
{
  "family": "baryons",
  "variant": "first_cooling_birth",
  "producer_route": [
    {
      "module": "core.first_stars",
      "function": "export_native_first_star_figures"
    }
  ],
  "input_contract": "Current source-bound original first-star-environments-v1; original cooling/birth numeric times match rows, exact namespace identities, saved interval and host ancestry brackets retained",
  "required_measurement": "Saved cooling/birth events with sufficient time resolution",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.first_stars.export_native_first_star_figures",
      "path": "core/first_stars.py",
      "line": 346,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 128 supported plotted entries.

![Original plot](baryons__first_cooling_birth.png)

Arrays: `baryons__first_cooling_birth.npz`; figure contract: `baryons__first_cooling_birth.json`.

- Recorded proper time [Myr] versus Event category: 0 cooling, 1 SSP birth [dimensionless]


### baryons / primordial_cloud_morphology

Distinguish thermal gas, effective EOS, seeded SSPs, actual birth events and spectral synthesis. Missing composition must remain missing.

Required contract: `Current source-bound primordial-cloud product; original measured proper Cartesian points/centroids and shape support, exact selected gas IDs/native namespace/epoch when available; empty unsupported physics retained`

```json
{
  "family": "baryons",
  "variant": "primordial_cloud_morphology",
  "producer_route": [
    {
      "module": "core.first_stars",
      "function": "export_native_cloud_figures"
    }
  ],
  "input_contract": "Current source-bound primordial-cloud product; original measured proper Cartesian points/centroids and shape support, exact selected gas IDs/native namespace/epoch when available; empty unsupported physics retained",
  "required_measurement": "Native cloud membership and resolved gas morphology",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.first_stars.export_native_cloud_figures",
      "path": "core/first_stars.py",
      "line": 447,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 65 supported plotted entries.

![Original plot](baryons__primordial_cloud_morphology.png)

Arrays: `baryons__primordial_cloud_morphology.npz`; figure contract: `baryons__primordial_cloud_morphology.json`.

- x [proper kpc] versus y [proper kpc]


### sidm / sidm_core_evolution

Read the interaction model, exact event/participant IDs, exposure clock, kernel support and independent controls.

Required contract: `Original source-bound native target/model/clock/identity products and unchanged dimensional support; no inferred detection from missing native support`

```json
{
  "family": "sidm",
  "variant": "sidm_core_evolution",
  "producer_route": [
    {
      "module": "core.sidm_products",
      "function": "export_native_sidm_history"
    }
  ],
  "input_contract": "Original source-bound native target/model/clock/identity products and unchanged dimensional support; no inferred detection from missing native support",
  "required_measurement": "Native scattering-enabled resolved core history",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.sidm_products.export_native_sidm_history",
      "path": "core/sidm_products.py",
      "line": 126,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 6 supported plotted entries.

![Original plot](sidm__sidm_core_evolution.png)

Arrays: `sidm__sidm_core_evolution.npz`; figure contract: `sidm__sidm_core_evolution.json`.

- Original elapsed time [Gyr] versus Central density [Msun / proper kpc^3]


### sidm / collapse_phase

Read the interaction model, exact event/participant IDs, exposure clock, kernel support and independent controls.

Required contract: `Original source-bound native target/model/clock/identity products and unchanged dimensional support; no inferred detection from missing native support`

```json
{
  "family": "sidm",
  "variant": "collapse_phase",
  "producer_route": [
    {
      "module": "core.sidm_products",
      "function": "export_native_sidm_history"
    }
  ],
  "input_contract": "Original source-bound native target/model/clock/identity products and unchanged dimensional support; no inferred detection from missing native support",
  "required_measurement": "Resolved density/dispersion evolution and collapse criterion",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.sidm_products.export_native_sidm_history",
      "path": "core/sidm_products.py",
      "line": 126,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 3 supported plotted entries.

![Original plot](sidm__collapse_phase.png)

Arrays: `sidm__collapse_phase.npz`; figure contract: `sidm__collapse_phase.json`.

- Original elapsed time [Gyr] versus Original phase category [dimensionless]


### sidm / scattering_rates

Read the interaction model, exact event/participant IDs, exposure clock, kernel support and independent controls.

Required contract: `Original source-bound native target/model/clock/identity products and unchanged dimensional support; no inferred detection from missing native support`

```json
{
  "family": "sidm",
  "variant": "scattering_rates",
  "producer_route": [
    {
      "module": "core.sidm_products",
      "function": "export_native_scattering_rates"
    }
  ],
  "input_contract": "Original source-bound native target/model/clock/identity products and unchanged dimensional support; no inferred detection from missing native support",
  "required_measurement": "Native event/rate ledger and physical cross-section model",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.sidm_products.export_native_scattering_rates",
      "path": "core/sidm_products.py",
      "line": 668,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 8 supported plotted entries.

![Original plot](sidm__scattering_rates.png)

Arrays: `sidm__scattering_rates.npz`; figure contract: `sidm__scattering_rates.json`.

- Actual interval end [Gyr] versus Native rare-event or optical-depth rate [Gyr^-1]
- Actual interval end [Gyr] versus Native rare-event or optical-depth rate [Gyr^-1]


### sidm / sidm_cdm_controls

Read the interaction model, exact event/participant IDs, exposure clock, kernel support and independent controls.

Required contract: `Original source-bound native target/model/clock/identity products and unchanged dimensional support; no inferred detection from missing native support`

```json
{
  "family": "sidm",
  "variant": "sidm_cdm_controls",
  "producer_route": [
    {
      "module": "core.sidm_products",
      "function": "export_native_sidm_matched"
    }
  ],
  "input_contract": "Original source-bound native target/model/clock/identity products and unchanged dimensional support; no inferred detection from missing native support",
  "required_measurement": "Matched SIDM/CDM phases, settings and catalogue selection",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.sidm_products.export_native_sidm_matched",
      "path": "core/sidm_products.py",
      "line": 278,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 6 supported plotted entries.

![Original plot](sidm__sidm_cdm_controls.png)

Arrays: `sidm__sidm_cdm_controls.npz`; figure contract: `sidm__sidm_cdm_controls.json`.

- Original elapsed time [Gyr] versus signed_difference [Msun / proper kpc^3]
- Original elapsed time [Gyr] versus relative_difference [dimensionless]


### sidm / sidm_density_dispersion_anisotropy

Read the interaction model, exact event/participant IDs, exposure clock, kernel support and independent controls.

Required contract: `Original source-bound native target/model/clock/identity products and unchanged dimensional support; no inferred detection from missing native support`

```json
{
  "family": "sidm",
  "variant": "sidm_density_dispersion_anisotropy",
  "producer_route": [
    {
      "module": "core.sidm_products",
      "function": "export_native_sidm_history"
    }
  ],
  "input_contract": "Original source-bound native target/model/clock/identity products and unchanged dimensional support; no inferred detection from missing native support",
  "required_measurement": "Resolved native SIDM profiles and velocity tensors",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.sidm_products.export_native_sidm_history",
      "path": "core/sidm_products.py",
      "line": 126,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 6 supported plotted entries.

![Original plot](sidm__sidm_density_dispersion_anisotropy.png)

Arrays: `sidm__sidm_density_dispersion_anisotropy.npz`; figure contract: `sidm__sidm_density_dispersion_anisotropy.json`.

- Measured shell radius [proper kpc] versus DM density [Msun / proper kpc^3]
- Measured shell radius [proper kpc] versus One-dimensional variance [(km / s)^2]
- Measured shell radius [proper kpc] versus Velocity anisotropy beta [dimensionless]


### sidm / sidm_shape_baryon_misalignment

Read the interaction model, exact event/participant IDs, exposure clock, kernel support and independent controls.

Required contract: `Original source-bound native target/model/clock/identity products and unchanged dimensional support; no inferred detection from missing native support`

```json
{
  "family": "sidm",
  "variant": "sidm_shape_baryon_misalignment",
  "producer_route": [
    {
      "module": "core.sidm_products",
      "function": "export_native_sidm_history"
    }
  ],
  "input_contract": "Original source-bound native target/model/clock/identity products and unchanged dimensional support; no inferred detection from missing native support",
  "required_measurement": "SIDM/baryon axes, angular degeneracy and shared frame",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.sidm_products.export_native_sidm_history",
      "path": "core/sidm_products.py",
      "line": 126,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 3 supported plotted entries.

![Original plot](sidm__sidm_shape_baryon_misalignment.png)

Arrays: `sidm__sidm_shape_baryon_misalignment.npz`; figure contract: `sidm__sidm_shape_baryon_misalignment.json`.

- Original elapsed time [Gyr] versus Unsigned component-axis angle [degrees]


### sidm / sidm_heat_flux

Read the interaction model, exact event/participant IDs, exposure clock, kernel support and independent controls.

Required contract: `Original source-bound native target/model/clock/identity products and unchanged dimensional support; no inferred detection from missing native support`

```json
{
  "family": "sidm",
  "variant": "sidm_heat_flux",
  "producer_route": [
    {
      "module": "core.sidm_products",
      "function": "export_native_sidm_history"
    }
  ],
  "input_contract": "Original source-bound native target/model/clock/identity products and unchanged dimensional support; no inferred detection from missing native support",
  "required_measurement": "Measured energy flux with sign convention and conservation",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.sidm_products.export_native_sidm_history",
      "path": "core/sidm_products.py",
      "line": 126,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 2 supported plotted entries.

![Original plot](sidm__sidm_heat_flux.png)

Arrays: `sidm__sidm_heat_flux.npz`; figure contract: `sidm__sidm_heat_flux.json`.

- Measured shell radius [proper kpc] versus Radial kinetic heat flux [erg / cm^2 / s]


### sidm / sidm_kernel_support

Read the interaction model, exact event/participant IDs, exposure clock, kernel support and independent controls.

Required contract: `Original source-bound native target/model/clock/identity products and unchanged dimensional support; no inferred detection from missing native support`

```json
{
  "family": "sidm",
  "variant": "sidm_kernel_support",
  "producer_route": [
    {
      "module": "core.sidm_products",
      "function": "export_native_sidm_history"
    }
  ],
  "input_contract": "Original source-bound native target/model/clock/identity products and unchanged dimensional support; no inferred detection from missing native support",
  "required_measurement": "Actual neighbor/kernel support and particle sampling",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.sidm_products.export_native_sidm_history",
      "path": "core/sidm_products.py",
      "line": 126,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 10 supported plotted entries.

![Original plot](sidm__sidm_kernel_support.png)

Arrays: `sidm__sidm_kernel_support.npz`; figure contract: `sidm__sidm_kernel_support.json`.

- Measured shell radius [proper kpc] versus Maximum coarse kernel fraction [dimensionless]
- Measured shell radius [proper kpc] versus Maximum omitted hazard bound [dimensionless]


### sidm / sidm_satellite_survival

Read the interaction model, exact event/participant IDs, exposure clock, kernel support and independent controls.

Required contract: `Original source-bound native target/model/clock/identity products and unchanged dimensional support; no inferred detection from missing native support`

```json
{
  "family": "sidm",
  "variant": "sidm_satellite_survival",
  "producer_route": [
    {
      "module": "core.sidm_products",
      "function": "export_native_sidm_extended"
    }
  ],
  "input_contract": "Original source-bound native target/model/clock/identity products and unchanged dimensional support; no inferred detection from missing native support",
  "required_measurement": "Persistent satellite IDs, disruption and completeness",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.sidm_products.export_native_sidm_extended",
      "path": "core/sidm_products.py",
      "line": 378,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 2 supported plotted entries.

![Original plot](sidm__sidm_satellite_survival.png)

Arrays: `sidm__sidm_satellite_survival.npz`; figure contract: `sidm__sidm_satellite_survival.json`.

- actual_time_gyr [Gyr] versus retained_original_bound_mass_fraction [dimensionless]
- actual_time_gyr [Gyr] versus current_bound_mass_msun [Msun]


### sidm / sidm_merger_centroid_offsets

Read the interaction model, exact event/participant IDs, exposure clock, kernel support and independent controls.

Required contract: `Original source-bound native target/model/clock/identity products and unchanged dimensional support; no inferred detection from missing native support`

```json
{
  "family": "sidm",
  "variant": "sidm_merger_centroid_offsets",
  "producer_route": [
    {
      "module": "core.sidm_products",
      "function": "export_native_sidm_centroids"
    }
  ],
  "input_contract": "Original source-bound native target/model/clock/identity products and unchanged dimensional support; no inferred detection from missing native support",
  "required_measurement": "Matched merging components, native centroids and errors",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.sidm_products.export_native_sidm_centroids",
      "path": "core/sidm_products.py",
      "line": 768,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 3 supported plotted entries.

![Original plot](sidm__sidm_merger_centroid_offsets.png)

Arrays: `sidm__sidm_merger_centroid_offsets.npz`; figure contract: `sidm__sidm_merger_centroid_offsets.json`.

- Actual saved time [Gyr] versus Centroid offset [proper kpc]
- Actual saved time [Gyr] versus Centroid offset [proper kpc]


### sidm / sidm_stream_profile_response

Read the interaction model, exact event/participant IDs, exposure clock, kernel support and independent controls.

Required contract: `Original source-bound native target/model/clock/identity products and unchanged dimensional support; no inferred detection from missing native support`

```json
{
  "family": "sidm",
  "variant": "sidm_stream_profile_response",
  "producer_route": [
    {
      "module": "core.sidm_products",
      "function": "export_native_sidm_extended"
    }
  ],
  "input_contract": "Original source-bound native target/model/clock/identity products and unchanged dimensional support; no inferred detection from missing native support",
  "required_measurement": "Matched native streams with resolved SIDM perturber profiles",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.sidm_products.export_native_sidm_extended",
      "path": "core/sidm_products.py",
      "line": 378,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 12 supported plotted entries.

![Original plot](sidm__sidm_stream_profile_response.png)

Arrays: `sidm__sidm_stream_profile_response.npz`; figure contract: `sidm__sidm_stream_profile_response.json`.

- original_bin_index [dimensionless] versus signed_difference [km / s]
- original_bin_index [dimensionless] versus floor_scaled_difference [dimensionless]


### sidm / sidm_lensing_h0_response

Read the interaction model, exact event/participant IDs, exposure clock, kernel support and independent controls.

Required contract: `Original source-bound native target/model/clock/identity products and unchanged dimensional support; no inferred detection from missing native support`

```json
{
  "family": "sidm",
  "variant": "sidm_lensing_h0_response",
  "producer_route": [
    {
      "module": "core.sidm_products",
      "function": "export_native_sidm_extended"
    }
  ],
  "input_contract": "Original source-bound native target/model/clock/identity products and unchanged dimensional support; no inferred detection from missing native support",
  "required_measurement": "Native mass/lightcone/lens products plus supported inference",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.sidm_products.export_native_sidm_extended",
      "path": "core/sidm_products.py",
      "line": 378,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 20 supported plotted entries.

![Original plot](sidm__sidm_lensing_h0_response.png)

Arrays: `sidm__sidm_lensing_h0_response.npz`; figure contract: `sidm__sidm_lensing_h0_response.json`.

- original_bin_index [dimensionless] versus signed_difference [dimensionless]
- original_bin_index [dimensionless] versus floor_scaled_difference [dimensionless]


### sidm / sidm_baryonic_degeneracy

Read the interaction model, exact event/participant IDs, exposure clock, kernel support and independent controls.

Required contract: `Original source-bound native target/model/clock/identity products and unchanged dimensional support; independently measured heldout calibration is mandatory`

```json
{
  "family": "sidm",
  "variant": "sidm_baryonic_degeneracy",
  "producer_route": [
    {
      "module": "core.sidm_products",
      "function": "export_native_sidm_extended"
    }
  ],
  "input_contract": "Original source-bound native target/model/clock/identity products and unchanged dimensional support; independently measured heldout calibration is mandatory",
  "required_measurement": "Matched baryonic interventions and SIDM parameter support",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.sidm_products.export_native_sidm_extended",
      "path": "core/sidm_products.py",
      "line": 378,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 6 supported plotted entries.

![Original plot](sidm__sidm_baryonic_degeneracy.png)

Arrays: `sidm__sidm_baryonic_degeneracy.npz`; figure contract: `sidm__sidm_baryonic_degeneracy.json`.

- mode_index [dimensionless] versus whitened_singular_value [dimensionless in declared
scaled parameter coordinates]
- parameter_index [dimensionless] versus mode_000 [dimensionless in declared scaled parameter
coordinates]
- parameter_index [dimensionless] versus mode_001 [dimensionless in declared scaled parameter
coordinates]


### hubble / hubble_method_comparison

Keep fitted observables, model calibration, covariance, mock/published data and privileged post-fit truth distinct.

Required contract: `Original completed source-bound cosmography/stream inference product, original physical prior/observable units, covariance, diagnostics, independent calibration and OOD support. See configs/research/analysis_branch_input_contracts.json.`

```json
{
  "family": "hubble",
  "variant": "hubble_method_comparison",
  "producer_route": [
    {
      "module": "core.expansion_products",
      "function": "export_hubble_figures"
    }
  ],
  "input_contract": "Original completed source-bound cosmography/stream inference product, original physical prior/observable units, covariance, diagnostics, independent calibration and OOD support. See configs/research/analysis_branch_input_contracts.json.",
  "required_measurement": "Actual method likelihoods/posteriors with common convention",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.expansion_products.export_hubble_figures",
      "path": "core/expansion_products.py",
      "line": 570,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 8 supported plotted entries.

![Original plot](hubble__hubble_method_comparison.png)

Arrays: `hubble__hubble_method_comparison.npz`; figure contract: `hubble__hubble_method_comparison.json`.

- Method index [dimensionless] versus Recovered minus original input [km/s/Mpc]


### hubble / distance_ladder

Keep fitted observables, model calibration, covariance, mock/published data and privileged post-fit truth distinct.

Required contract: `Original completed source-bound cosmography/stream inference product, original physical prior/observable units, covariance, diagnostics, independent calibration and OOD support. See configs/research/analysis_branch_input_contracts.json.`

```json
{
  "family": "hubble",
  "variant": "distance_ladder",
  "producer_route": [
    {
      "module": "core.expansion_products",
      "function": "export_hubble_figures"
    }
  ],
  "input_contract": "Original completed source-bound cosmography/stream inference product, original physical prior/observable units, covariance, diagnostics, independent calibration and OOD support. See configs/research/analysis_branch_input_contracts.json.",
  "required_measurement": "Observed/calibrated distance-ladder data and likelihood",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.expansion_products.export_hubble_figures",
      "path": "core/expansion_products.py",
      "line": 570,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 4 supported plotted entries.

![Original plot](hubble__distance_ladder.png)

Arrays: `hubble__distance_ladder.npz`; figure contract: `hubble__distance_ladder.json`.

- Ladder estimate index [dimensionless] versus H0 [km/s/Mpc]


### hubble / bao_cmb_background

Keep fitted observables, model calibration, covariance, mock/published data and privileged post-fit truth distinct.

Required contract: `Original completed source-bound cosmography/stream inference product, original physical prior/observable units, covariance, diagnostics, independent calibration and OOD support. See configs/research/analysis_branch_input_contracts.json.`

```json
{
  "family": "hubble",
  "variant": "bao_cmb_background",
  "producer_route": [
    {
      "module": "core.expansion_products",
      "function": "export_hubble_figures"
    }
  ],
  "input_contract": "Original completed source-bound cosmography/stream inference product, original physical prior/observable units, covariance, diagnostics, independent calibration and OOD support. See configs/research/analysis_branch_input_contracts.json.",
  "required_measurement": "Pinned BAO/CMB likelihood and background model",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.expansion_products.export_hubble_figures",
      "path": "core/expansion_products.py",
      "line": 570,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 26 supported plotted entries.

![Original plot](hubble__bao_cmb_background.png)

Arrays: `hubble__bao_cmb_background.npz`; figure contract: `hubble__bao_cmb_background.json`.

- Observed redshift [dimensionless] versus DH_over_rs [dimensionless]
- Observed redshift [dimensionless] versus DM_over_rs [dimensionless]
- Observed redshift [dimensionless] versus DV_over_rs [dimensionless]


### hubble / time_delay_posteriors

Keep fitted observables, model calibration, covariance, mock/published data and privileged post-fit truth distinct.

Required contract: `Original completed source-bound cosmography/stream inference product, original physical prior/observable units, covariance, diagnostics, independent calibration and OOD support. See configs/research/analysis_branch_input_contracts.json.`

```json
{
  "family": "hubble",
  "variant": "time_delay_posteriors",
  "producer_route": [
    {
      "module": "core.expansion_products",
      "function": "export_hubble_figures"
    }
  ],
  "input_contract": "Original completed source-bound cosmography/stream inference product, original physical prior/observable units, covariance, diagnostics, independent calibration and OOD support. See configs/research/analysis_branch_input_contracts.json.",
  "required_measurement": "Actual time-delay lens likelihood, nuisance and posterior samples",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.expansion_products.export_hubble_figures",
      "path": "core/expansion_products.py",
      "line": 570,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 1680 supported plotted entries.

![Original plot](hubble__time_delay_posteriors.png)

Arrays: `hubble__time_delay_posteriors.npz`; figure contract: `hubble__time_delay_posteriors.json`.

- H0 [km/s/Mpc] versus Empirical cumulative probability [dimensionless]
- Time-delay distance [Mpc] versus Empirical cumulative probability [dimensionless]


### inference / actual_posterior

Read actual chain/model diagnostics, priors, heldout lineage and OOD decisions. A rendered posterior can still be unusable.

Required contract: `Original completed source-bound cosmography/stream inference product, original physical prior/observable units, covariance, diagnostics, independent calibration and OOD support. See configs/research/analysis_branch_input_contracts.json.`

```json
{
  "family": "inference",
  "variant": "actual_posterior",
  "producer_route": [
    {
      "module": "core.flagship_inference",
      "function": "export_inference_figures"
    }
  ],
  "input_contract": "Original completed source-bound cosmography/stream inference product, original physical prior/observable units, covariance, diagnostics, independent calibration and OOD support. See configs/research/analysis_branch_input_contracts.json.",
  "required_measurement": "Actual validated likelihood/simulator posterior samples",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_inference.export_inference_figures",
      "path": "core/flagship_inference.py",
      "line": 151,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 256 supported plotted entries.

![Original plot](inference__actual_posterior.png)

Arrays: `inference__actual_posterior.npz`; figure contract: `inference__actual_posterior.json`.

- x_0 [dimensionless] versus y_0 [dimensionless]
- x_1 [dimensionless] versus y_1 [dimensionless]


### inference / posterior_predictive

Read actual chain/model diagnostics, priors, heldout lineage and OOD decisions. A rendered posterior can still be unusable.

Required contract: `Original completed source-bound cosmography/stream inference product, original physical prior/observable units, covariance, diagnostics, independent calibration and OOD support. See configs/research/analysis_branch_input_contracts.json.`

```json
{
  "family": "inference",
  "variant": "posterior_predictive",
  "producer_route": [
    {
      "module": "core.flagship_inference",
      "function": "export_inference_figures"
    }
  ],
  "input_contract": "Original completed source-bound cosmography/stream inference product, original physical prior/observable units, covariance, diagnostics, independent calibration and OOD support. See configs/research/analysis_branch_input_contracts.json.",
  "required_measurement": "Held-out forward predictions and observed-data contract",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_inference.export_inference_figures",
      "path": "core/flagship_inference.py",
      "line": 151,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 8 supported plotted entries.

![Original plot](inference__posterior_predictive.png)

Arrays: `inference__posterior_predictive.npz`; figure contract: `inference__posterior_predictive.json`.

- x_replication [dimensionless] versus y_replication [dimensionless]


### inference / sbc_ranks

Read actual chain/model diagnostics, priors, heldout lineage and OOD decisions. A rendered posterior can still be unusable.

Required contract: `Original completed source-bound cosmography/stream inference product, original physical prior/observable units, covariance, diagnostics, independent calibration and OOD support. See configs/research/analysis_branch_input_contracts.json.`

```json
{
  "family": "inference",
  "variant": "sbc_ranks",
  "producer_route": [
    {
      "module": "core.flagship_inference",
      "function": "export_inference_figures"
    }
  ],
  "input_contract": "Original completed source-bound cosmography/stream inference product, original physical prior/observable units, covariance, diagnostics, independent calibration and OOD support. See configs/research/analysis_branch_input_contracts.json.",
  "required_measurement": "Independent calibration simulations, truths and ranks",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_inference.export_inference_figures",
      "path": "core/flagship_inference.py",
      "line": 151,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 899 supported plotted entries.

![Original plot](inference__sbc_ranks.png)

Arrays: `inference__sbc_ranks.npz`; figure contract: `inference__sbc_ranks.json`.

- x_0 [dimensionless] versus y_0 [dimensionless]


### inference / coverage

Read actual chain/model diagnostics, priors, heldout lineage and OOD decisions. A rendered posterior can still be unusable.

Required contract: `Original completed source-bound cosmography/stream inference product, original physical prior/observable units, covariance, diagnostics, independent calibration and OOD support. See configs/research/analysis_branch_input_contracts.json.`

```json
{
  "family": "inference",
  "variant": "coverage",
  "producer_route": [
    {
      "module": "core.flagship_inference",
      "function": "export_inference_figures"
    }
  ],
  "input_contract": "Original completed source-bound cosmography/stream inference product, original physical prior/observable units, covariance, diagnostics, independent calibration and OOD support. See configs/research/analysis_branch_input_contracts.json.",
  "required_measurement": "Held-out truths, credible regions and sampling uncertainty",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_inference.export_inference_figures",
      "path": "core/flagship_inference.py",
      "line": 151,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 3 supported plotted entries.

![Original plot](inference__coverage.png)

Arrays: `inference__coverage.npz`; figure contract: `inference__coverage.json`.

- x_coverage [dimensionless] versus y_coverage [dimensionless]
- x_lower [dimensionless] versus y_lower [dimensionless]
- x_upper [dimensionless] versus y_upper [dimensionless]


### inference / ood_support

Read actual chain/model diagnostics, priors, heldout lineage and OOD decisions. A rendered posterior can still be unusable.

Required contract: `Original completed source-bound cosmography/stream inference product, original physical prior/observable units, covariance, diagnostics, independent calibration and OOD support. See configs/research/analysis_branch_input_contracts.json.`

```json
{
  "family": "inference",
  "variant": "ood_support",
  "producer_route": [
    {
      "module": "core.flagship_inference",
      "function": "export_inference_figures"
    }
  ],
  "input_contract": "Original completed source-bound cosmography/stream inference product, original physical prior/observable units, covariance, diagnostics, independent calibration and OOD support. See configs/research/analysis_branch_input_contracts.json.",
  "required_measurement": "Training/support domain and measured out-of-domain scores",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_inference.export_inference_figures",
      "path": "core/flagship_inference.py",
      "line": 151,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 2 supported plotted entries.

![Original plot](inference__ood_support.png)

Arrays: `inference__ood_support.npz`; figure contract: `inference__ood_support.json`.

- x_distance [dimensionless] versus y_distance [dimensionless]
- x_outside [dimensionless] versus y_outside [dimensionless]


### inference / graph_baseline_gain

Read actual chain/model diagnostics, priors, heldout lineage and OOD decisions. A rendered posterior can still be unusable.

Required contract: `Original completed source-bound cosmography/stream inference product, original physical prior/observable units, covariance, diagnostics, independent calibration and OOD support. See configs/research/analysis_branch_input_contracts.json.`

```json
{
  "family": "inference",
  "variant": "graph_baseline_gain",
  "producer_route": [
    {
      "module": "core.flagship_inference",
      "function": "export_inference_figures"
    }
  ],
  "input_contract": "Original completed source-bound cosmography/stream inference product, original physical prior/observable units, covariance, diagnostics, independent calibration and OOD support. See configs/research/analysis_branch_input_contracts.json.",
  "required_measurement": "Same held-out lineage split, baseline and graph-model metric",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_inference.export_inference_figures",
      "path": "core/flagship_inference.py",
      "line": 151,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

**No retained example.** Supply the named physical inputs and execute the original producer.


### convergence / mass_resolution

Vary one declared control on matched targets. Retain unsupported bins and paired cross covariance; one smoke contrast is not general convergence.

Required contract: `comparison compare_campaign contract, hash-bound measurements, physical bin_coordinate`

```json
{
  "family": "convergence",
  "variant": "mass_resolution",
  "producer_route": [
    {
      "module": "core.flagship_products",
      "function": "export_convergence_figures"
    }
  ],
  "input_contract": "comparison compare_campaign contract, hash-bound measurements, physical bin_coordinate",
  "required_measurement": "Same physical targets across mass levels, matched epochs",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_products.export_convergence_figures",
      "path": "core/flagship_products.py",
      "line": 818,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 8 supported plotted entries.

![Original plot](convergence__mass_resolution.png)

Arrays: `convergence__mass_resolution.npz`; figure contract: `convergence__mass_resolution.json`.

- saved elapsed time [native elapsed time L/V] versus Difference [native length L]


### convergence / softening

Vary one declared control on matched targets. Retain unsupported bins and paired cross covariance; one smoke contrast is not general convergence.

Required contract: `comparison compare_campaign contract, hash-bound measurements, physical bin_coordinate`

```json
{
  "family": "convergence",
  "variant": "softening",
  "producer_route": [
    {
      "module": "core.flagship_products",
      "function": "export_convergence_figures"
    }
  ],
  "input_contract": "comparison compare_campaign contract, hash-bound measurements, physical bin_coordinate",
  "required_measurement": "Resolved same-target softening ladder and observables",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_products.export_convergence_figures",
      "path": "core/flagship_products.py",
      "line": 818,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 8 supported plotted entries.

![Original plot](convergence__softening.png)

Arrays: `convergence__softening.npz`; figure contract: `convergence__softening.json`.

- saved elapsed time [native elapsed time L/V] versus Difference [native acceleration V^2/L]


### convergence / timestep

Vary one declared control on matched targets. Retain unsupported bins and paired cross covariance; one smoke contrast is not general convergence.

Required contract: `comparison compare_campaign contract, hash-bound measurements, physical bin_coordinate`

```json
{
  "family": "convergence",
  "variant": "timestep",
  "producer_route": [
    {
      "module": "core.flagship_products",
      "function": "export_convergence_figures"
    }
  ],
  "input_contract": "comparison compare_campaign contract, hash-bound measurements, physical bin_coordinate",
  "required_measurement": "Resolved same-target integration-step ladder and observables",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_products.export_convergence_figures",
      "path": "core/flagship_products.py",
      "line": 818,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 10 supported plotted entries.

![Original plot](convergence__timestep.png)

Arrays: `convergence__timestep.npz`; figure contract: `convergence__timestep.json`.

- saved elapsed time [native elapsed time L/V] versus Difference [dimensionless]


### convergence / output_cadence

Vary one declared control on matched targets. Retain unsupported bins and paired cross covariance; one smoke contrast is not general convergence.

Required contract: `comparison compare_campaign contract, hash-bound measurements, physical bin_coordinate`

```json
{
  "family": "convergence",
  "variant": "output_cadence",
  "producer_route": [
    {
      "module": "core.flagship_products",
      "function": "export_convergence_figures"
    }
  ],
  "input_contract": "comparison compare_campaign contract, hash-bound measurements, physical bin_coordinate",
  "required_measurement": "Dense saved outputs and measured thinning/interpolation error",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_products.export_convergence_figures",
      "path": "core/flagship_products.py",
      "line": 818,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 10 supported plotted entries.

![Original plot](convergence__output_cadence.png)

Arrays: `convergence__output_cadence.npz`; figure contract: `convergence__output_cadence.json`.

- saved elapsed time [native elapsed time L/V] versus Difference [native length L]


### convergence / start_redshift

Vary one declared control on matched targets. Retain unsupported bins and paired cross covariance; one smoke contrast is not general convergence.

Required contract: `comparison compare_campaign contract, hash-bound measurements, physical bin_coordinate`

```json
{
  "family": "convergence",
  "variant": "start_redshift",
  "producer_route": [
    {
      "module": "core.flagship_products",
      "function": "export_convergence_figures"
    }
  ],
  "input_contract": "comparison compare_campaign contract, hash-bound measurements, physical bin_coordinate",
  "required_measurement": "Same phases and final observables across start epochs",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_products.export_convergence_figures",
      "path": "core/flagship_products.py",
      "line": 818,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 4 supported plotted entries.

![Original plot](convergence__start_redshift.png)

Arrays: `convergence__start_redshift.npz`; figure contract: `convergence__start_redshift.json`.

- output redshift [dimensionless] versus Difference [km/s]


### convergence / finder_tree_variation

Vary one declared control on matched targets. Retain unsupported bins and paired cross covariance; one smoke contrast is not general convergence.

Required contract: `comparison compare_campaign contract, hash-bound measurements, physical bin_coordinate`

```json
{
  "family": "convergence",
  "variant": "finder_tree_variation",
  "producer_route": [
    {
      "module": "core.flagship_products",
      "function": "export_convergence_figures"
    }
  ],
  "input_contract": "comparison compare_campaign contract, hash-bound measurements, physical bin_coordinate",
  "required_measurement": "Same native snapshots through finder/tree alternatives",
  "source_contract": "Original source-bound numerical product/receipt; exact IDs, native saved epoch, physical/statistical unit conventions and target/selection retained.",
  "units": "Original producer per-dataset units; renderer rejects mismatching axes or undeclared conversion.",
  "uncertainty_support": "Original measured covariance/uncertainty and boolean support; missing runtime measurements are explicit unresolved/pending and never invented.",
  "renderer": {
    "module": "core.research.display",
    "function": "export_scientific_suite"
  },
  "artifacts": [
    "PNG",
    "SVG",
    "PDF",
    "physical_arrays.npz",
    "figure.json",
    "gallery.json"
  ],
  "recipe_contract": "Use producer-specific request or campaign constructor; commands.py flagship-products, export-hubble-figures, export-inference-figures and dedicated preparers retain original input contracts.",
  "implementation_route": [
    {
      "name": "core.flagship_products.export_convergence_figures",
      "path": "core/flagship_products.py",
      "line": 818,
      "role": "producer"
    },
    {
      "name": "core.research.display.export_scientific_suite",
      "path": "core/research/display.py",
      "line": 1263,
      "role": "renderer"
    }
  ]
}
```

Archived example: 4 supported plotted entries.

![Original plot](convergence__finder_tree_variation.png)

Arrays: `convergence__finder_tree_variation.npz`; figure contract: `convergence__finder_tree_variation.json`.

- actual scale factor [dimensionless] versus Difference [count]


## Replay a selected archived figure

Set `RESEARCH2_LAB_FIGURE` to a `family/variant` pair before starting the kernel,
or edit `selected` in the next cell. The default opens the certified
impact-geometry control when present. The cell verifies the archive hashes,
displays the figure and axis contracts, and prints each measured array's shape,
type and finite numeric range. The implementation chain identifies the
producer and renderer source locations in your checkout.


In [ ]:
from hashlib import sha256
selected = os.environ.get('RESEARCH2_LAB_FIGURE', 'encounters/impact_velocity_geometry')
recipes = {g['family'] + '/' + g['variant']: g for g in catalogue['graphs']}
recipe = recipes[selected]
example = recipe['reference']
for step in recipe['implementation_route']:
    print(step['role'], step['name'], step['path'], 'line', step['line'])
if example is None:
    print('No retained example. Required input:', recipe['input_contract'])
else:
    directory = ROOT / 'optional_tests/reference_gallery'
    for key, field in [('arrays', 'arrays_sha256'), ('image', 'image_sha256'),
                       ('manifest', 'original_manifest_sha256')]:
        path = directory / example[key]
        assert sha256(path.read_bytes()).hexdigest() == example[field], path
    display(Image(filename=str(directory / example['image'])))
    for axis in example['axes']:
        print(axis.get('x_label', ''), '|', axis.get('y_label', ''), '|', axis.get('z_label', ''))
    print(example['supported_entries'], 'supported entries across plotted series')
    with np.load(directory / example['arrays'], allow_pickle=False) as arrays:
        for name in arrays.files:
            value = arrays[name]
            print(name, value.shape, value.dtype, end=' ')
            if value.dtype.kind == 'b':
                print(int(value.sum()), 'true')
            elif value.dtype.kind in 'iuf':
                finite = value[np.isfinite(value)]
                print((finite.min().item(), finite.max().item()) if finite.size else 'no finite values')
            else:
                print()
    print('Full figure contract:', directory / example['manifest'])
